# Modélisation – H7 : prévoir le prix d'une vente plutôt que la médiane d'un territoire
**Contributeurs :** Cécile · **Date :** 07/11/2026, mis à jour le 10/11/2026

Notebook construit à partir du template commun de l'équipe. Il passe par `commun.py` pour tout ce qui doit être identique d'une hypothèse à l'autre : la table, le masque des lignes, les fenêtres, la naïve, et pour les ventes `evaluer_ventes`, `tester_ventes` (évaluation B) et `evaluer_panier` (évaluation C).

**Question :** un modèle par vente sépare-t-il l'effet du bien et celui du marché, et fait-il mieux que les modèles agrégés à information égale ? Il ne devrait pas mieux anticiper le retournement. Les hypothèses du niveau 2 (volume et élan, capacité d'achat) et les trimestres précédents sont ensuite ajoutés au réseau, pour voir s'ils aident à l'anticiper.

## 0. Paramètres de l'hypothèse
- `HYPOTHESE` : préfixe du nom des modèles dans les tableaux.
- `ANNEE_DEBUT_H7` : première année d'apprentissage. Le modèle par vente apprend sur la période longue (à partir de 2017), pour voir plusieurs phases du marché.
- Les blocs de variables sont fixés après le constat rapide (section 2).

In [7]:
HYPOTHESE = "H7"
ANNEE_DEBUT_H7 = 2017      # période longue : apprentissage à partir des ventes de 2017

## 1. Import et chargement
- `charger_table()` : la table de modélisation, avec le masque commun ; elle donne à chaque vente les variables de son sous-bassin (prix N-1, bassin de vie, variables de territoire).
- `charger_ventes(table)` : les ventes de l'échantillon (2014-2024), gardées seulement dans les sous-bassins de la table pour leur année (au moins 20 ventes en N et en N-1). Le prix médian de N et le nombre de ventes de N ne sont pas ajoutés : ils sont inconnus au moment de prévoir. La cible d'une vente est `prix_m2`.
- `charger_ventes_brutes()` : les mêmes ventes sans rattachement, pour le panier fixe (évaluation C).
- `fenetres(..., annee_debut=2017)` : une fenêtre par année jugée (2023, 2024), apprentissage de 2017 à l'année d'avant. L'échantillon 2025 n'est pas chargé.

In [8]:
import numpy as np
import pandas as pd
from scipy import stats

import commun
from commun import charger_table, fenetres, charger_ventes, charger_ventes_brutes

pd.set_option("display.width", 200)

table = charger_table()
ventes_brutes = charger_ventes_brutes()
ventes = charger_ventes(table)
FENETRES_TABLE = fenetres(table, annee_debut=ANNEE_DEBUT_H7)     # lignes agrégées (évaluation C)
FENETRES = fenetres(ventes, annee_debut=ANNEE_DEBUT_H7)          # ventes (apprentissage, évaluation B)

# Ventes gardées par année : celles des sous-bassins sous le seuil ou à case vide sont retirées
garde = pd.DataFrame({"ventes de l'échantillon": ventes_brutes.groupby("anneemut").size(),
                      "ventes rattachées": ventes.groupby("annee").size()})
garde["part gardée (%)"] = (garde["ventes rattachées"] / garde["ventes de l'échantillon"] * 100).round(1)
print(garde.loc[ANNEE_DEBUT_H7:].to_string())

# Contrôles : prix N-1 et bassin de vie connus pour chaque vente
assert ventes["prix_n1"].notna().all() and ventes["BV2022"].notna().all()
assert ventes["annee"].max() <= commun.ANNEES_JUGEES[-1], "aucune vente de 2025 ne doit être chargée"

for a, (app, ev) in FENETRES.items():
    print(f"Fenêtre {a} : apprentissage {sorted(int(x) for x in app['annee'].unique())} "
          f"({len(app)} ventes) | jugée {a} ({len(ev)} ventes)")

      ventes de l'échantillon  ventes rattachées  part gardée (%)
2017                    83523            82725.0             99.0
2018                    83304            82555.0             99.1
2019                    91533            90756.0             99.2
2020                    89204            88529.0             99.2
2021                   102718           101956.0             99.3
2022                    98340            97692.0             99.3
2023                    78477            77828.0             99.2
2024                    71808            71127.0             99.1
Fenêtre 2023 : apprentissage [2017, 2018, 2019, 2020, 2021, 2022] (544213 ventes) | jugée 2023 (77828 ventes)
Fenêtre 2024 : apprentissage [2017, 2018, 2019, 2020, 2021, 2022, 2023] (622041 ventes) | jugée 2024 (71127 ventes)


## 2. Constat rapide : les variables du bien
Avant de choisir les transformations, on regarde les variables candidates du bien sur les années d'apprentissage de la première fenêtre (2017-2022) : les années jugées ne sont pas regardées. Par type de bien : cases vides, valeurs nulles, distribution, nombre de pièces, et cohérence entre surface et nombre de pièces.

Variables candidates : surface bâtie (`sbati`), surface du terrain (`sterr`), nombre de pièces (`nb_pieces`, 5 = 5 ou plus), nombre de dépendances (`nblocdep`), mois de la vente (`moismut`), coordonnées de la parcelle (`geompar_x`, `geompar_y`).

In [9]:
app_constat = FENETRES[commun.ANNEES_JUGEES[0]][0]
BIEN_CANDIDATES = ["sbati", "sterr", "nb_pieces", "nblocdep", "moismut", "geompar_x", "geompar_y"]
par_type = app_constat.groupby("type_bien")

print("Types :", app_constat[BIEN_CANDIDATES].dtypes.astype(str).to_dict())

# Cases vides et valeurs nulles (%), par type de bien
print("\nCases vides (%)")
print(par_type[BIEN_CANDIDATES].apply(lambda g: g.isna().mean() * 100).T.round(2).to_string())
print("\nValeurs à 0 (%)")
print(par_type[BIEN_CANDIDATES].apply(lambda g: (g == 0).mean() * 100).T.round(2).to_string())

# Distribution, par type de bien
for type_bien, e in par_type:
    print(f"\n{type_bien} ({len(e)} ventes)")
    print(e[BIEN_CANDIDATES].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.round(1).to_string())

# Répartition du nombre de pièces (%) et surface médiane par nombre de pièces
print("\nNombre de pièces (% des ventes de chaque type)")
print(pd.crosstab(app_constat["nb_pieces"], app_constat["type_bien"], normalize="columns").mul(100).round(1).to_string())
print("\nSurface bâtie médiane (m²) par nombre de pièces")
print(app_constat.pivot_table(index="nb_pieces", columns="type_bien", values="sbati", aggfunc="median").to_string())

Types : {'sbati': 'float64', 'sterr': 'float64', 'nb_pieces': 'int64', 'nblocdep': 'int64', 'moismut': 'int64', 'geompar_x': 'float64', 'geompar_y': 'float64'}

Cases vides (%)
type_bien  appartement  maison
sbati             0.00    0.00
sterr             0.00    0.00
nb_pieces         0.00    0.00
nblocdep          0.00    0.00
moismut           0.00    0.00
geompar_x         0.01    0.02
geompar_y         0.01    0.02

Valeurs à 0 (%)
type_bien  appartement  maison
sbati             0.00    0.00
sterr            98.33    4.41
nb_pieces         0.00    0.00
nblocdep         38.52   83.88
moismut           0.00    0.00
geompar_x         0.00    0.00
geompar_y         0.00    0.00

appartement (228029 ventes)
              count       mean       std        min         1%         5%        50%        95%        99%         max
sbati      228029.0       58.3      27.8       10.0       15.0       21.0       56.0      104.0      147.0       735.0
sterr      228029.0       12.7     663.5   

### 2.1 Préparation des variables du bien
Transformations décidées d'après le constat :
- **Surface bâtie** en log : un mètre carré de plus ne compte pas pareil dans un studio et dans une grande maison.
- **Terrain** : mis à 0 pour les appartements (la surface relevée est celle de la copropriété, pas du logement) ; pour les maisons, borné au 99e centile de 2017-2022, pour que les quelques ventes avec des terres agricoles ne pèsent pas sur le modèle, puis en log(1 + x) (beaucoup de petites valeurs et des zéros).
- **Nombre de pièces** : une indicatrice par valeur (1, 2, 3, 4, 5 ou plus).
- **Dépendances** : une indicatrice par catégorie (0, 1, 2, 3 ou plus).
- Le **mois de la vente** n'est pas retenu : on ne cherche pas la saisonnalité.
- Les **coordonnées de la parcelle** ne font pas partie du bloc du bien. Elles forment un bloc à part, testé en H7v1b : l'hypothèse est qu'elles sont redondantes avec le sous-bassin. Les coordonnées hors de la France métropolitaine et les cases vides (moins de 0,03 % des ventes) sont remplacées par la médiane des coordonnées du sous-bassin sur 2017-2022.

Les colonnes indicatrices sont écrites une à une, pour être identiques dans tous les jeux (apprentissage, année jugée, panier fixe), même si une catégorie y est absente. Les mêmes transformations sont appliquées aux ventes brutes, qui servent au panier fixe.

In [10]:
# Seuil du terrain des maisons : 99e centile de 2017-2022 (années d'apprentissage de la première fenêtre)
SEUIL_STERR = app_constat.loc[app_constat["type_bien"] == "maison", "sterr"].quantile(0.99)

# Coordonnées (Lambert 93) : limites de la France métropolitaine, Corse comprise
BORNES_X, BORNES_Y = (100_000, 1_250_000), (6_030_000, 7_120_000)

COLONNES_PIECES = ["pieces_1", "pieces_2", "pieces_3", "pieces_4", "pieces_5_plus"]
COLONNES_DEP = ["dep_0", "dep_1", "dep_2", "dep_3_plus"]
BLOC_BIEN = ["maison", "log_sbati", "log1p_sterr"] + COLONNES_PIECES + COLONNES_DEP
BLOC_COORD = ["geompar_x", "geompar_y"]


def coordonnees_douteuses(v):
    """Vrai pour une coordonnée vide ou hors de la France métropolitaine."""
    return (~v["geompar_x"].between(*BORNES_X)) | (~v["geompar_y"].between(*BORNES_Y))


# Médiane des coordonnées de chaque sous-bassin, sur les coordonnées valides de 2017-2022
valides = app_constat[~coordonnees_douteuses(app_constat)]
CENTRES = valides.groupby("sous_bassin_de_vie")[BLOC_COORD].median()


def preparer_bien(v):
    """Variables du bien et coordonnées : transformations ligne à ligne, seuils fixés d'après 2017-2022."""
    v = v.copy()
    v["log_sbati"] = np.log(v["sbati"])
    terrain = v["sterr"].where(v["type_bien"] == "maison", 0).clip(upper=SEUIL_STERR)
    v["log1p_sterr"] = np.log1p(terrain)
    for k, col in enumerate(COLONNES_PIECES, start=1):          # 5 = 5 pièces ou plus
        v[col] = (v["nb_pieces"] == k).astype(int)
    dependances = v["nblocdep"].clip(upper=3)                   # 3 = 3 dépendances ou plus
    for k, col in enumerate(COLONNES_DEP):
        v[col] = (dependances == k).astype(int)
    douteuses = coordonnees_douteuses(v)
    for c in BLOC_COORD:
        v.loc[douteuses, c] = v.loc[douteuses, "sous_bassin_de_vie"].map(CENTRES[c])
    return v


print("Coordonnées remplacées (ventes 2017-2024) :", int(coordonnees_douteuses(ventes).sum()))
ventes = preparer_bien(ventes)
ventes_brutes = preparer_bien(ventes_brutes)                    # pour le panier fixe (évaluation C)
FENETRES = fenetres(ventes, annee_debut=ANNEE_DEBUT_H7)        # fenêtres refaites avec les nouvelles colonnes

# Contrôles
assert ventes[BLOC_BIEN + BLOC_COORD].notna().all().all(), "case vide dans le bloc du bien ou les coordonnées"
assert not coordonnees_douteuses(ventes).any(), "coordonnée encore hors de la France métropolitaine"
assert (ventes[COLONNES_PIECES].sum(axis=1) == 1).all() and (ventes[COLONNES_DEP].sum(axis=1) == 1).all(), \
    "chaque vente doit avoir exactement une catégorie de pièces et une de dépendances"
assert (ventes.loc[ventes["type_bien"] == "appartement", "log1p_sterr"] == 0).all(), "terrain des appartements non nul"

maisons = app_constat["type_bien"] == "maison"
print(f"Seuil du terrain des maisons : {SEUIL_STERR:,.0f} m² | maisons bornées en 2017-2022 : "
      f"{(app_constat.loc[maisons, 'sterr'] > SEUIL_STERR).mean() * 100:.1f} %")
app_bien = FENETRES[commun.ANNEES_JUGEES[0]][0]
print(app_bien.groupby("type_bien")[BLOC_BIEN].mean().T.round(3).to_string())

Coordonnées remplacées (ventes 2017-2024) : 152
Seuil du terrain des maisons : 26,704 m² | maisons bornées en 2017-2022 : 1.0 %
type_bien      appartement  maison
maison               0.000   1.000
log_sbati            3.953   4.546
log1p_sterr          0.000   6.284
pieces_1             0.166   0.021
pieces_2             0.297   0.064
pieces_3             0.312   0.192
pieces_4             0.172   0.330
pieces_5_plus        0.054   0.393
dep_0                0.385   0.839
dep_1                0.426   0.118
dep_2                0.152   0.034
dep_3_plus           0.037   0.009


### 2.2 Lien avec le prix, redondances
Sur les années d'apprentissage de la première fenêtre (2017-2022), par type de bien, rho de Spearman entre chaque variable du bien et :
- le **prix au m²** de la vente : ce lien mélange l'effet du bien et celui du lieu (les petits logements sont plus fréquents dans les villes chères) ;
- l'**écart au prix médian N-1 du sous-bassin** (prix au m² / prix N-1 − 1) : le niveau du lieu est en grande partie retiré, il reste ce que le bien explique. C'est ce que le modèle par vente doit apprendre en plus du sous-bassin.

Redondances : couples de variables du bloc dont |rho| dépasse 0,7. Avec plusieurs centaines de milliers de ventes, tout rho non nul serait significatif : les rho décrivent la force des liens, ils ne se testent pas.

In [11]:
ecart_n1 = app_bien["prix_m2"] / app_bien["prix_n1"] - 1
VARIABLES_LIEN = [v for v in BLOC_BIEN if v != "maison"]       # maison est constante dans chaque type de bien

liens = []
for type_bien, e in app_bien.groupby("type_bien"):
    for v in VARIABLES_LIEN:
        if e[v].nunique() < 2:                                  # ex. terrain des appartements, toujours à 0
            continue
        liens.append({"type de bien": type_bien, "variable": v,
                      "rho avec le prix au m²": stats.spearmanr(e[v], e["prix_m2"]).statistic,
                      "rho avec l'écart au prix N-1": stats.spearmanr(e[v], ecart_n1.loc[e.index]).statistic})
liens = pd.DataFrame(liens).pivot(index="variable", columns="type de bien")
print(liens.reindex(VARIABLES_LIEN).round(2).to_string())

couples = []
for type_bien, e in app_bien.groupby("type_bien"):
    rho = e[VARIABLES_LIEN].corr(method="spearman")
    for i, a in enumerate(VARIABLES_LIEN):
        for b in VARIABLES_LIEN[i + 1:]:
            if abs(rho.loc[a, b]) > 0.7:
                couples.append({"type de bien": type_bien, "variable 1": a, "variable 2": b,
                                "rho": round(rho.loc[a, b], 2)})
print(pd.DataFrame(couples).to_string(index=False) if couples else "Aucun couple au-delà de 0,7")

              rho avec le prix au m²        rho avec l'écart au prix N-1       
type de bien             appartement maison                  appartement maison
variable                                                                       
log_sbati                      -0.21  -0.04                        -0.18  -0.11
log1p_sterr                      NaN  -0.10                          NaN   0.15
pieces_1                        0.14  -0.04                         0.13  -0.02
pieces_2                        0.08  -0.04                         0.08   0.00
pieces_3                       -0.06  -0.03                        -0.05   0.02
pieces_4                       -0.13   0.05                        -0.13   0.05
pieces_5_plus                  -0.05   0.00                        -0.06  -0.06
dep_0                           0.02  -0.06                        -0.04  -0.06
dep_1                          -0.02   0.06                         0.00   0.04
dep_2                          -0.00   0

**Comment lire :** le rho de Spearman va de −1 à +1 ; son signe donne le sens du lien, sa valeur absolue la force (environ 0,1 : faible ; 0,3 : moyen ; 0,5 et plus : fort). Exemple : pour les appartements, le rho entre `log_sbati` et le prix au m² vaut −0,21 : plus le logement est grand, plus le m² est bon marché, avec un lien faible à moyen. Il vaut encore −0,18 avec l'écart au prix N-1 : l'effet tient à l'intérieur d'un même sous-bassin, c'est un effet du bien et non du lieu.

**Constat :**

- Les liens entre les variables du bien et le prix sont faibles : le plus fort est celui de la surface des appartements (−0,21 avec le prix au m², −0,18 avec l'écart au prix N-1). Le nombre de pièces des appartements suit la surface : les studios ont un m² plus cher (+0,14 avec le prix, +0,13 avec l'écart), les 4 pièces un m² moins cher (−0,13). Les dépendances n'ont presque aucun lien avec le prix (|rho| ≤ 0,06).
- Pour les maisons, le terrain change de signe : −0,10 avec le prix au m², +0,15 avec l'écart au prix N-1. Les grands terrains se trouvent surtout là où les prix sont bas, mais dans un même sous-bassin, plus de terrain donne un prix au m² plus élevé : le prix de vente comprend le terrain et il est divisé par la seule surface bâtie. La surface bâtie des maisons va dans le même sens que celle des appartements, avec un lien plus faible sur le prix (−0,04) et plus net à l'intérieur d'un sous-bassin (−0,11).
- Seul couple au-delà de 0,7 : `dep_0` et `dep_1` pour les maisons (−0,84), un effet mécanique de l'encodage (84 % des maisons sans dépendance, 12 % avec une seule). Les deux variables sont gardées.
- Ces rho ne mesurent que des liens monotones, une variable à la fois ; le réseau peut apprendre des combinaisons (surface et lieu). On attend un gain modeste du bien par rapport au sous-bassin seul.

## 3. Fonctions du notebook
Chaque modèle est lancé en une ligne : `lancer(nom, famille, variables, **réglages)`.
- `famille` : `"keras"` (gabarit `gabarit_keras.py`) ou `"elastic"` (Elastic Net par vente, ci-dessous) ;
- `variables` : les colonnes données au modèle, dont `sous_bassin_de_vie` quand le lieu est utilisé ;
- un seul modèle pour les deux types de bien (`maison` est une variable), poids `w_h` dans l'apprentissage et dans les mesures.

Pour chaque année jugée (2023, puis 2024), `lancer` :
1. apprend sur les ventes de 2017 à l'année d'avant ;
2. prévoit le prix au m² de chaque vente de l'année jugée : **évaluation B** (`evaluer_ventes`, `tester_ventes`), naïve = prix médian N-1 du sous-bassin de la vente ;
3. prévoit, pour l'année jugée, le prix des biens vendus l'année d'avant (panier fixe) : la médiane pondérée de ces prévisions est comparée au prix médian réel du sous-bassin, comme pour les modèles agrégés : **évaluation C** (`evaluer_panier`).

**Elastic Net par vente.** Le modèle pont (H4) est un Elastic Net appris sur la table agrégée : son gabarit n'est pas modifié, la même méthode est adaptée ici aux ventes.
- Il reçoit exactement les mêmes colonnes que la version du réseau à laquelle il est comparé.
- La cible est le log du prix au m², remise en €/m² à la prévision ; les variables sont mises à l'échelle (moyennes et écarts-types calculés sur les ventes d'apprentissage).
- Un modèle linéaire ne peut pas utiliser directement 1 700 sous-bassins comme catégories. Chaque sous-bassin est remplacé par un code appris : la moyenne du log du prix de ses ventes d'apprentissage (`TargetEncoder`). Ce code est calculé avec un découpage interne en 5 parts : une vente ne sert jamais à calculer son propre code. Un sous-bassin inconnu reçoit la moyenne générale.
- La grille du modèle pont (`alpha` et `l1_ratio`) est évaluée sur la dernière année d'apprentissage, comme l'arrêt du réseau ; le meilleur réglage est ensuite réappris sur toute la fenêtre. Pas de validation croisée sur tous les plis : avec plus de 500 000 ventes, le temps de calcul serait trop long.

In [12]:
import time
import warnings
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.preprocessing import StandardScaler, TargetEncoder

from commun import evaluer_ventes, tester_ventes, panier_fixe, evaluer_panier
from gabarit_pont import GRILLE_EN
from gabarit_keras import prevoir_keras

warnings.filterwarnings("ignore", category=ConvergenceWarning)   # alpha très petits : convergence lente, sans effet sur la prévision
resultats, tests, ajustes, previsions = [], [], {}, {}


class ElasticNetVentes(RegressorMixin, BaseEstimator):
    """Elastic Net par vente : apprend log(prix au m²), rend des €/m².

    Mêmes colonnes que la version Keras comparée (variables déjà transformées en log dans le notebook).
    Le sous-bassin est remplacé par un code appris (TargetEncoder, découpage interne en 5 parts).
    La colonne 'annee', si elle est présente, n'est jamais une variable.
    """

    def __init__(self, colonne_lieu="sous_bassin_de_vie", alpha=1e-3, l1_ratio=0.5, random_state=commun.GRAINE):
        self.colonne_lieu = colonne_lieu
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.random_state = random_state

    def _variables(self, X, code):
        Xn = X[self.variables_].astype(float)      # mêmes colonnes qu'à l'apprentissage, dans le même ordre
        return Xn if code is None else Xn.assign(code_lieu=code)

    def fit(self, X, y, sample_weight=None):
        cible = np.log(np.asarray(y, dtype=float))
        self.avec_lieu_ = self.colonne_lieu in X.columns
        self.variables_ = [c for c in X.columns if c not in ("annee", self.colonne_lieu)]
        code = None
        if self.avec_lieu_:
            self.encodeur_ = TargetEncoder(target_type="continuous",
                                           cv=KFold(5, shuffle=True, random_state=self.random_state))
            code = self.encodeur_.fit_transform(X[[self.colonne_lieu]], cible).ravel()
        Xn = self._variables(X, code)
        self.echelle_ = StandardScaler().fit(Xn, sample_weight=sample_weight)
        # precompute=True : les liens entre variables sont calculés une seule fois (petit tableau variables x variables) ;
        # chaque tour de l'ajustement ne relit plus les centaines de milliers de ventes. Mêmes coefficients, beaucoup plus rapide
        self.modele_ = ElasticNet(alpha=self.alpha, l1_ratio=self.l1_ratio, max_iter=50_000, precompute=True)
        self.modele_.fit(self.echelle_.transform(Xn), cible, sample_weight=sample_weight)
        self.coefs_ = pd.Series(self.modele_.coef_, index=Xn.columns)     # coefficients standardisés
        return self

    def predict(self, X):
        code = self.encodeur_.transform(X[[self.colonne_lieu]]).ravel() if self.avec_lieu_ else None
        Z = self.echelle_.transform(self._variables(X, code))
        return pd.Series(np.exp(self.modele_.predict(Z)), index=X.index)


def prevoir_elastic_ventes(app, ev, variables, grille=None):
    """Évalue la grille sur la dernière année d'apprentissage, réapprend sur toute la fenêtre, prévoit ev."""
    grille = GRILLE_EN if grille is None else grille
    colonnes = list(variables) + ["annee"]
    derniere = app["annee"].max()
    ent, arr = app[app["annee"] < derniere], app[app["annee"] == derniere]
    essais = []
    for reglage in ParameterGrid(grille):
        m = ElasticNetVentes(**reglage).fit(ent[colonnes], ent["prix_m2"], sample_weight=ent["w_h"])
        mae = np.average((arr["prix_m2"] - m.predict(arr[colonnes])).abs(), weights=arr["w_h"])
        essais.append({**reglage, "MAE année d'arrêt (€/m²)": mae})
    essais = pd.DataFrame(essais).sort_values("MAE année d'arrêt (€/m²)")
    meilleur = {k: essais.iloc[0][k] for k in grille}
    modele = ElasticNetVentes(**meilleur).fit(app[colonnes], app["prix_m2"], sample_weight=app["w_h"])
    modele.essais_ = essais
    return modele.predict(ev[colonnes]), modele


def lancer(nom, famille, variables, **reglages):
    """Un modèle par fenêtre (un seul pour les deux types de bien), puis évaluations B et C.

    Réseau Keras : les réglages optimisés (REGLAGES_KERAS, section 3.1) s'appliquent à toutes les versions ;
    un réglage passé à lancer garde la priorité (ex. colonne_embedding=None). Si la cible retenue est
    l'évolution et que la version a le prix N-1, le réseau apprend le rapport prix / prix N-1.
    """
    assert famille in ("keras", "elastic")
    # Relancer une cellule remplace les résultats du modèle au lieu de les ajouter une seconde fois
    resultats[:] = [r for r in resultats if not r["modèle"].startswith(nom + " (")]
    tests[:] = [t for t in tests if not t["modèle"].startswith(nom + " (")]
    if famille == "keras":
        reglages = {**REGLAGES_KERAS, **reglages}
        if EVOLUTION and "log_prix_n1" in variables:
            reglages.setdefault("reference", "log_prix_n1")
    for annee, (app, ev) in FENETRES.items():
        debut = time.time()
        panier = panier_fixe(ventes_brutes, table, annee)        # biens vendus en N-1, datés de N
        if famille == "keras":
            prevision, m = prevoir_keras(app, ev, variables, "prix_m2", poids="w_h", **reglages)
        else:
            prevision, m = prevoir_elastic_ventes(app, ev, variables, **reglages)
        prevision_panier = m.predict(panier)
        ajustes[(nom, annee)] = m
        previsions[(nom, annee)] = (prevision, prevision_panier)
        evaluer_ventes(nom, ev, prevision, resultats)
        tester_ventes(nom, ev, prevision, tests)
        evaluer_panier(nom, panier, prevision_panier, table, resultats, tests)
        print(f"{nom} | fenêtre {annee} : {(time.time() - debut) / 60:.1f} min")

    res = pd.DataFrame(resultats)
    res = res[res["modèle"].str.startswith(nom + " (")][["modèle", "type de bien", "année jugée", "MAE (€/m²)",
                                                         "MAE naïve (€/m²)", "gain sur la naïve (%)",
                                                         "bassins où le modèle gagne (%)"]]
    res["évaluation"] = res.pop("modèle").str.extract(r"\((B|C)", expand=False)
    print(nom, f"({len(variables)} variables)", reglages if famille == "keras" else "")
    print(res.round(1).to_string(index=False))
    return nom

I0000 00:00:1795601324.596613   10992 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795601324.605351   10992 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795601325.465160   10992 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795601327.801470   10992 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_E

### 3.1 Réglages du réseau
Les réglages des réseaux Keras viennent de `H7_optimisation.py` (fichier `meilleurs_reglages_keras.json`). Ils ont été choisis sur une année de validation (2022) qui n'est jamais une année jugée, avec des ventes de 2017 à 2021 seulement : aucune année jugée n'a servi au choix.
- **Cible** : quand la version dispose du prix N-1, le réseau apprend le rapport entre le prix de la vente et le prix médian N-1 de son sous-bassin, puis prévoit prix N-1 × rapport. Le niveau de chaque territoire vient ainsi du prix N-1 : le réseau ne ramène plus les prix élevés vers le milieu (appartements les plus chers). Les versions sans prix N-1 (H7v1 à H7v2b) apprennent toujours le prix.
- **Perte** : erreur absolue (MAE), la même mesure que celle qui juge les modèles.
- **Réglages** : couches, dropout, taux d'apprentissage et taille de l'embedding retenus par une grille complète.
- Un seul réseau pour les deux types de bien : un réseau par type a été testé et n'apporte rien de stable.

In [13]:
import json

with open(commun.DATA_OUT / "meilleurs_reglages_keras.json", encoding="utf-8") as f:
    OPTIMISATION_KERAS = json.load(f)["reglages"]

assert OPTIMISATION_KERAS["organisation"] == "commun", "H7 n'utilise qu'un réseau pour les deux types de bien"
EVOLUTION = OPTIMISATION_KERAS["cible"] == "evolution"
REGLAGES_KERAS = {k: OPTIMISATION_KERAS[k] for k in ["dropout", "learning_rate", "dim_embedding", "perte"]}
REGLAGES_KERAS["couches"] = tuple(OPTIMISATION_KERAS["couches"])

print("Cible :", "évolution par rapport au prix N-1" if EVOLUTION else "prix", "| réglages :", REGLAGES_KERAS)

Cible : évolution par rapport au prix N-1 | réglages : {'dropout': 0.1, 'learning_rate': 0.0003, 'dim_embedding': 4, 'perte': 'mae', 'couches': (128, 64, 32)}


## 4. Modèles
D'abord le « où », puis le « quand ». Chaque version Keras est comparée à un Elastic Net par vente qui dispose des mêmes informations : l'Elastic Net est le modèle retenu au niveau 1 (H3) et pour le modèle pont (H4). Tous les modèles sont gardés, y compris ceux qui perdent.

### 4.1 Référence du « où » : Elastic Net, sous-bassin + bien
Mêmes informations que H7v1 : le lieu (sous-bassin codé) et les variables du bien, sans prix N-1 ni indicateur de la phase du marché.

In [14]:
REFERENCE_OU = lancer("H7 – Elastic Net – sous-bassin + bien", "elastic", ["sous_bassin_de_vie"] + BLOC_BIEN)

# Réglages essayés sur la dernière année d'apprentissage (3 meilleurs) et coefficients standardisés du modèle réappris
for annee in commun.ANNEES_JUGEES:
    m = ajustes[(REFERENCE_OU, annee)]
    print(f"\nFenêtre {annee}")
    print(m.essais_.head(3).round(5).to_string(index=False))
    print(m.coefs_.round(3).to_string())

H7 – Elastic Net – sous-bassin + bien | fenêtre 2023 : 0.2 min
H7 – Elastic Net – sous-bassin + bien | fenêtre 2024 : 0.2 min
H7 – Elastic Net – sous-bassin + bien (13 variables) 
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       844.8             732.6                  -15.3                            31.2          B
      maison         2023       861.0             774.2                  -11.2                            30.1          B
 appartement         2023       497.9             166.2                 -199.5                            15.2          C
      maison         2023       458.9             111.7                 -310.8                             8.0          C
 appartement         2024       841.1             744.9                  -12.9                            35.8          B
      maison         2024       788.8             740.6                   -6.5          

**Comment lire :** le tableau donne, pour chaque année jugée et chaque type de bien, l'erreur moyenne du modèle (MAE, en €/m²), celle de la naïve et le gain sur la naïve (négatif : le modèle fait moins bien). L'évaluation B juge chaque vente (naïve = prix médian N-1 de son sous-bassin) ; l'évaluation C juge le prix médian de chaque sous-bassin, comme les modèles agrégés. Exemple : en 2023, pour les appartements, l'Elastic Net se trompe en moyenne de 845 €/m² par vente, contre 733 €/m² pour la naïve, soit un gain de −15,3 % ; il ne fait mieux que la naïve que dans 31 % des sous-bassins.

Sous ce tableau, pour chaque fenêtre : les trois meilleurs réglages essayés sur la dernière année d'apprentissage, puis les coefficients standardisés du modèle réappris (variables centrées-réduites, prix en log : ils se comparent entre eux, pas à des €/m²).

**Résultats :**

- Sans information sur le moment, l'Elastic Net perd contre la naïve partout : de −6,5 à −15,3 % en évaluation B, de −178 à −311 % en évaluation C.
- Le réglage retient un `alpha` très petit (0,00001 en 2023, le plus petit de la grille ; 0,00032 en 2024), et les trois meilleurs réglages ne diffèrent que de quelques centimes d'€/m² : avec 13 variables et environ 450 000 ventes, la pénalité ne sert presque à rien.
- Les coefficients sont presque identiques d'une fenêtre à l'autre. Après le lieu (`code_lieu`, 0,58), le terrain (+0,16 et +0,15) et la surface (−0,12) pèsent le plus, comme l'annonçait le constat (section 2.2) ; viennent ensuite l'absence de dépendance (`dep_0`, −0,05) et le type de bien (`maison`, −0,03).
- À surface égale, le nombre de pièces change de sens par rapport au constat : les petits nombres de pièces ont un coefficient négatif (`pieces_1` −0,02), les grands un coefficient positif (`pieces_5_plus` +0,02). Le rho mélangeait pièces et surface ; le modèle sépare les deux effets.
- `pieces_3` et `dep_1` valent 0 : les indicatrices d'un même groupe font toujours 1 au total, et l'Elastic Net en retire une de lui-même.

### 4.2 H7v1 : réseau Keras, sous-bassin (embedding) + bien
Mêmes informations que la référence 4.1 : le lieu, appris par un embedding (un vecteur de 8 valeurs par sous-bassin), et les variables du bien. Réglages du gabarit par défaut.

**Biais des prévisions (évaluation B).** Les modèles du « où » ne savent pas en quelle année ils prévoient : ils apprennent un niveau moyen de la période 2017 à l'année d'avant. Le biais est l'écart moyen entre prévision et prix réel, en % (moyenne des log, pondérée par `w_h`) : négatif, le modèle sous-estime.

In [15]:
H7V1 = lancer("H7v1 – Keras – sous-bassin + bien", "keras", ["sous_bassin_de_vie"] + BLOC_BIEN)


def biais(noms):
    """Écart moyen prévision / prix réel (%) par modèle, année jugée et type de bien (évaluation B)."""
    lignes = []
    for nom in noms:
        for annee, (app, ev) in FENETRES.items():
            ecart = np.log(previsions[(nom, annee)][0] / ev["prix_m2"])
            for type_bien, e in ev.groupby("type_bien"):
                lignes.append({"modèle": nom, "année jugée": annee, "type de bien": type_bien,
                               "biais (%)": (np.exp(np.average(ecart.loc[e.index], weights=e["w_h"])) - 1) * 100})
    return pd.DataFrame(lignes).pivot_table(index=["modèle", "type de bien"], columns="année jugée",
                                            values="biais (%)").round(1)


biais([REFERENCE_OU, H7V1])

E0000 00:00:1795601348.296917   10992 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


H7v1 – Keras – sous-bassin + bien | fenêtre 2023 : 3.7 min
H7v1 – Keras – sous-bassin + bien | fenêtre 2024 : 2.4 min
H7v1 – Keras – sous-bassin + bien (13 variables) {'dropout': 0.1, 'learning_rate': 0.0003, 'dim_embedding': 4, 'perte': 'mae', 'couches': (128, 64, 32)}
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       802.8             732.6                   -9.6                            43.0          B
      maison         2023       791.6             774.2                   -2.2                            53.6          B
 appartement         2023       454.6             166.2                 -173.5                            19.5          C
      maison         2023       342.5             111.7                 -206.6                            19.3          C
 appartement         2024       803.9             744.9                   -7.9                            45.9         

année jugée                                         2023  2024
modèle                                type de bien            
H7 – Elastic Net – sous-bassin + bien appartement   -8.9  -5.6
                                      maison       -11.4  -6.8
H7v1 – Keras – sous-bassin + bien     appartement   -9.8  -6.5
                                      maison        -7.5  -2.4

In [16]:
for a in commun.ANNEES_JUGEES:
    m = ajustes[(H7V1, a)]
    print(a, "| meilleure époque :", m.meilleure_epoque_, "| époques faites :", len(m.historique_["perte_arret"]),
          "| perte d'arrêt :", [round(p, 3) for p in m.historique_["perte_arret"][:20]])

2023 | meilleure époque : 34 | époques faites : 49 | perte d'arrêt : [0.515, 0.489, 0.484, 0.483, 0.486, 0.49, 0.481, 0.486, 0.488, 0.484, 0.484, 0.483, 0.482, 0.483, 0.48, 0.482, 0.484, 0.481, 0.476, 0.478]
2024 | meilleure époque : 16 | époques faites : 31 | perte d'arrêt : [0.479, 0.471, 0.468, 0.468, 0.463, 0.462, 0.463, 0.46, 0.461, 0.46, 0.46, 0.458, 0.457, 0.456, 0.461, 0.455, 0.457, 0.46, 0.459, 0.457]


**Résultats :**

- H7v1 perd encore contre la naïve, sauf de justesse pour les maisons en 2024 : de −9,6 à +1,6 % en évaluation B (le +1,6 % reste sous le seuil de 3 %), de −98 à −207 % en évaluation C.
- À informations égales, le réseau fait mieux que l'Elastic Net dans les huit cas. En évaluation B : 803 contre 845 €/m² pour les appartements en 2023, 792 contre 861 pour les maisons ; 804 contre 841 et 729 contre 789 en 2024. En évaluation C, l'écart est plus net pour les maisons (343 contre 459 €/m² en 2023, 256 contre 360 en 2024). Les tests de la section 5.2 disent si ces écarts dépassent le hasard.
- Les deux modèles sous-estiment les prix, de 2 à 11 % : sans prix N-1 ni indicateur de la phase du marché, ils ne savent pas en quelle année ils prévoient. Pour un lieu et un bien, ils prévoient un niveau moyen de la période d'apprentissage, alors que les prix ont fortement monté entre 2017 et 2022. Le biais baisse en 2024 pour les deux modèles, après la baisse des prix et avec 2023 dans l'apprentissage ; il est plus faible pour le réseau sur les maisons (−7,5 puis −2,4 %), proche de l'Elastic Net sur les appartements (−9,8 puis −6,5 %).
- En évaluation B, ce que le modèle gagne en tenant compte du bien est plus que perdu par l'erreur de niveau, sauf pour les maisons en 2024 où les deux se compensent presque ; en évaluation C, l'effet du bien disparaît dans la médiane et seule l'erreur de niveau reste.
- Arrêt du réseau (cellule ci-dessus) : la meilleure époque est la 34e en 2023 et la 16e en 2024. La perte d'arrêt baisse surtout pendant les trois ou quatre premières époques (de 0,515 à 0,483 en 2023, de 0,479 à 0,468 en 2024), puis de quelques millièmes seulement. Ce n'est pas l'entraînement qui limite le modèle, c'est l'information qui manque.

### 4.3 H7v1b : réseau Keras, sous-bassin (embedding) + bien + coordonnées
H7v1 avec, en plus, les coordonnées de la parcelle. Hypothèse : la localisation fine est redondante avec le sous-bassin ; si c'est le cas, H7v1b ne fait pas mieux que H7v1.

In [17]:
H7V1B = lancer("H7v1b – Keras – sous-bassin + bien + coordonnées", "keras",
               ["sous_bassin_de_vie"] + BLOC_BIEN + BLOC_COORD)
biais([REFERENCE_OU, H7V1, H7V1B])

H7v1b – Keras – sous-bassin + bien + coordonnées | fenêtre 2023 : 5.8 min
H7v1b – Keras – sous-bassin + bien + coordonnées | fenêtre 2024 : 3.9 min
H7v1b – Keras – sous-bassin + bien + coordonnées (15 variables) {'dropout': 0.1, 'learning_rate': 0.0003, 'dim_embedding': 4, 'perte': 'mae', 'couches': (128, 64, 32)}
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       792.6             732.6                   -8.2                            42.2          B
      maison         2023       790.5             774.2                   -2.1                            53.3          B
 appartement         2023       442.0             166.2                 -165.9                            18.4          C
      maison         2023       343.4             111.7                 -207.4                            19.6          C
 appartement         2024       793.4             744.9                   

année jugée                                                    2023  2024
modèle                                           type de bien            
H7 – Elastic Net – sous-bassin + bien            appartement   -8.9  -5.6
                                                 maison       -11.4  -6.8
H7v1 – Keras – sous-bassin + bien                appartement   -9.8  -6.5
                                                 maison        -7.5  -2.4
H7v1b – Keras – sous-bassin + bien + coordonnées appartement   -9.5  -7.4
                                                 maison        -7.3  -2.1

**Résultats :** les coordonnées n'apportent presque rien au sous-bassin. En évaluation B, l'erreur de H7v1b est inférieure à celle de H7v1 d'environ 10 €/m² pour les appartements (793 contre 803 en 2023, 793 contre 804 en 2024, soit 1,3 %) et de 1 à 9 €/m² pour les maisons ; en évaluation C, les écarts restent de quelques €/m² et vont dans les deux sens. Le biais ne change presque pas (de −2 à −10 %, contre −2 à −10 % pour H7v1). Pour les maisons en 2024, H7v1b bat la naïve de 2,8 %, juste sous le seuil de 3 %. Les tests de la section 5.2 diront si l'écart avec H7v1 dépasse le hasard ; de toute façon, il est faible. Ce qui manque n'est pas une localisation plus fine, mais l'information sur le moment.

### 4.4 Le territoire : variables retenues au niveau 1
Variables de territoire retenues au niveau 1 (H3 globale) : le parc de logements (9 indicateurs), la vacance (LOVAC) et la densité. Ce sont des photos fixes : pour un sous-bassin, elles ont la même valeur chaque année. Les variables asymétriques passent en log (mêmes règles que les autres notebooks), dans la table et dans les ventes : le panier fixe de l'évaluation C reprend ses variables dans la table.

In [18]:
PARC = ["taux_vacance", "part_rsec", "part_apparts", "log_nb_logements", "part_moins40m2",
        "part_avant1946", "part_2006_2019", "anciennete_moyenne", "part_sans_voiture"]
VACANCE = ["taux_vacance_complete"]
DENSITE = ["densite_population"]
LOG_TERRITOIRE = [v for v in PARC + VACANCE + DENSITE
                  if v in ["densite_population", "part_sans_voiture", "part_moins40m2", "part_rsec", "part_apparts"]]
BLOC_TERRITOIRE = ["log_" + v if v in LOG_TERRITOIRE else v for v in PARC + VACANCE + DENSITE]


def preparer_territoire(t):
    """Variables de territoire asymétriques en log (transformation ligne à ligne)."""
    t = t.copy()
    assert (t[LOG_TERRITOIRE] > 0).all().all(), "valeur nulle ou négative dans une variable passée en log"
    for v in LOG_TERRITOIRE:
        t["log_" + v] = np.log(t[v])
    return t


table = preparer_territoire(table)              # pour le panier fixe (évaluation C)
ventes = preparer_territoire(ventes)
FENETRES = fenetres(ventes, annee_debut=ANNEE_DEBUT_H7)
assert ventes[BLOC_TERRITOIRE].notna().all().all(), "case vide dans le bloc du territoire"
print(len(BLOC_TERRITOIRE), "variables de territoire :", BLOC_TERRITOIRE)

11 variables de territoire : ['taux_vacance', 'log_part_rsec', 'log_part_apparts', 'log_nb_logements', 'log_part_moins40m2', 'part_avant1946', 'part_2006_2019', 'anciennete_moyenne', 'log_part_sans_voiture', 'taux_vacance_complete', 'log_densite_population']


### 4.5 H7v2a : réseau Keras, sous-bassin (embedding) + bien + territoire
H7v1 avec, en plus, les variables de territoire. Question : le territoire ajoute-t-il quelque chose à l'identité du sous-bassin ? L'embedding apprend déjà ce qui est propre et constant pour chaque sous-bassin : on attend un gain faible.

In [19]:
H7V2A = lancer("H7v2a – Keras – sous-bassin + bien + territoire", "keras",
               ["sous_bassin_de_vie"] + BLOC_BIEN + BLOC_TERRITOIRE)
biais([REFERENCE_OU, H7V1, H7V1B, H7V2A])

H7v2a – Keras – sous-bassin + bien + territoire | fenêtre 2023 : 3.4 min
H7v2a – Keras – sous-bassin + bien + territoire | fenêtre 2024 : 3.4 min
H7v2a – Keras – sous-bassin + bien + territoire (24 variables) {'dropout': 0.1, 'learning_rate': 0.0003, 'dim_embedding': 4, 'perte': 'mae', 'couches': (128, 64, 32)}
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       799.7             732.6                   -9.2                            42.3          B
      maison         2023       788.3             774.2                   -1.8                            54.6          B
 appartement         2023       456.8             166.2                 -174.8                            17.7          C
      maison         2023       349.5             111.7                 -212.9                            17.9          C
 appartement         2024       792.0             744.9                   -6.

année jugée                                                    2023  2024
modèle                                           type de bien            
H7 – Elastic Net – sous-bassin + bien            appartement   -8.9  -5.6
                                                 maison       -11.4  -6.8
H7v1 – Keras – sous-bassin + bien                appartement   -9.8  -6.5
                                                 maison        -7.5  -2.4
H7v1b – Keras – sous-bassin + bien + coordonnées appartement   -9.5  -7.4
                                                 maison        -7.3  -2.1
H7v2a – Keras – sous-bassin + bien + territoire  appartement  -10.4  -8.1
                                                 maison        -7.6  -2.8

**Résultats :** le territoire ajoute peu à l'identité du sous-bassin. En évaluation B, l'erreur de H7v2a est inférieure à celle de H7v1 de 3 €/m² en 2023 (appartements comme maisons) et d'environ 12 €/m² en 2024 (792 contre 804 €/m² pour les appartements, 718 contre 729 pour les maisons). En évaluation C, les écarts vont dans les deux sens (+2 et +7 €/m² en 2023, −15 et −5 en 2024). Pour les maisons en 2024, le gain sur la naïve atteint 3,1 %, juste au-dessus du seuil ; l'année ne compte comme réussie que si l'IC et la rang-bisériale le confirment (section 5.2). Le biais reste du même ordre (de −3 à −10 %). Les variables de territoire sont des photos fixes, et l'embedding apprend déjà ce qui est propre et constant pour chaque sous-bassin.

### 4.6 H7v2b : réseau Keras, bien + territoire, sans embedding
Le sous-bassin n'est plus donné au réseau : les variables de territoire le remplacent. Question : le territoire suffit-il à décrire le lieu ? Si H7v2b fait aussi bien que H7v1, le lieu se résume à ces 11 variables ; s'il fait nettement moins bien, l'identité du sous-bassin porte une information que le territoire ne décrit pas.

In [20]:
H7V2B = lancer("H7v2b – Keras – bien + territoire, sans embedding", "keras",
               BLOC_BIEN + BLOC_TERRITOIRE, colonne_embedding=None)
biais([REFERENCE_OU, H7V1, H7V1B, H7V2A, H7V2B])

H7v2b – Keras – bien + territoire, sans embedding | fenêtre 2023 : 6.5 min
H7v2b – Keras – bien + territoire, sans embedding | fenêtre 2024 : 4.3 min
H7v2b – Keras – bien + territoire, sans embedding (23 variables) {'dropout': 0.1, 'learning_rate': 0.0003, 'dim_embedding': 4, 'perte': 'mae', 'couches': (128, 64, 32), 'colonne_embedding': None}
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       814.7             732.6                  -11.2                            39.6          B
      maison         2023       808.5             774.2                   -4.4                            49.6          B
 appartement         2023       482.0             166.2                 -190.0                            19.0          C
      maison         2023       374.7             111.7                 -235.5                            16.8          C
 appartement         2024       823.3       

année jugée                                                     2023  2024
modèle                                            type de bien            
H7 – Elastic Net – sous-bassin + bien             appartement   -8.9  -5.6
                                                  maison       -11.4  -6.8
H7v1 – Keras – sous-bassin + bien                 appartement   -9.8  -6.5
                                                  maison        -7.5  -2.4
H7v1b – Keras – sous-bassin + bien + coordonnées  appartement   -9.5  -7.4
                                                  maison        -7.3  -2.1
H7v2a – Keras – sous-bassin + bien + territoire   appartement  -10.4  -8.1
                                                  maison        -7.6  -2.8
H7v2b – Keras – bien + territoire, sans embedding appartement   -9.5  -7.6
                                                  maison        -7.3  -2.9

**Résultats :** sans l'identité du sous-bassin, les 11 variables de territoire décrivent une bonne part du lieu, mais pas tout. En évaluation B, H7v2b fait moins bien que H7v1 dans les quatre cas, de 12 à 19 €/m² (1,5 à 2,4 %), et moins bien que H7v2a de 15 à 31 €/m². L'écart est plus net en évaluation C (de 27 à 47 €/m² de plus que H7v1). Le biais reste du même ordre (de −3 à −10 %). L'embedding garde donc un avantage régulier, même s'il est petit : le parc de logements, la vacance et la densité décrivent l'essentiel de ce qui distingue les sous-bassins, pas tout. La version suivante part de H7v2a, la meilleure des deux.

### 4.7 Le « quand » : prix N-1
On ajoute le prix médian N-1 du sous-bassin, la seule information des modèles sur le niveau récent du marché local. Il entre en log (`log_prix_n1`), dans la table et dans les ventes, pour le réseau comme pour l'Elastic Net. C'est la structure du modèle pont : le log du prix expliqué par le log du prix N-1.

In [21]:
# Prix N-1 en log, dans la table (panier fixe) et dans les ventes
table["log_prix_n1"] = np.log(table["prix_n1"])
ventes["log_prix_n1"] = np.log(ventes["prix_n1"])
FENETRES = fenetres(ventes, annee_debut=ANNEE_DEBUT_H7)

#### Export des ventes préparées (pour l'optimisation du réseau)
Les ventes, avec toutes les variables préparées (bien, territoire, prix N-1 en log), sont enregistrées dans `ventes_h7.parquet`, avec la liste des variables de H7v3 dans `variables_h7v3.json`. Le notebook `H7_optimisation.py` les lit : il règle le réseau sur exactement les mêmes variables que H7v3, sans recopier la préparation.

In [22]:
import json

VARIABLES_H7V3 = ["sous_bassin_de_vie"] + BLOC_BIEN + BLOC_TERRITOIRE + ["log_prix_n1"]
assert "valeurfonc" not in ventes.columns, "valeurfonc ne doit pas figurer dans l'export : avec sbati, elle redonne la cible"
ventes.to_parquet(commun.DATA_OUT / "ventes_h7.parquet", index=False)
with open(commun.DATA_OUT / "variables_h7v3.json", "w", encoding="utf-8") as f:
    json.dump(VARIABLES_H7V3, f, ensure_ascii=False, indent=1)

taille = (commun.DATA_OUT / "ventes_h7.parquet").stat().st_size / 1e6
print(f"Exporté : ventes_h7.parquet ({len(ventes)} ventes, {ventes.shape[1]} colonnes, {taille:.0f} Mo)")
print(f"Variables de H7v3 : {len(VARIABLES_H7V3)}")

Exporté : ventes_h7.parquet (693168 ventes, 68 colonnes, 61 Mo)
Variables de H7v3 : 25


### 4.8 Référence du « quand » : Elastic Net, sous-bassin + bien + territoire + prix N-1
Mêmes colonnes que H7v3. Avec le log du prix N-1 comme variable, apprendre le log du prix revient à apprendre l'évolution par rapport au prix N-1 : le coefficient du log du prix N-1 s'en charge. Un coefficient proche de 1 veut dire que le prix suit celui de l'année précédente, comme pour la naïve (même lecture que pour le modèle pont, H4).

In [23]:
REFERENCE_QUAND = lancer("H7 – Elastic Net – sous-bassin + bien + territoire + prix N-1", "elastic",
                         ["sous_bassin_de_vie"] + BLOC_BIEN + BLOC_TERRITOIRE + ["log_prix_n1"])

# Réglage retenu et coefficient du log du prix N-1, remis à l'échelle d'origine (comme coef_prix_n1_ du gabarit pont)
for annee in commun.ANNEES_JUGEES:
    m = ajustes[(REFERENCE_QUAND, annee)]
    echelle_n1 = m.echelle_.scale_[list(m.coefs_.index).index("log_prix_n1")]
    print(f"Fenêtre {annee} | réglage : alpha {m.alpha:.0e}, l1_ratio {m.l1_ratio} "
          f"| coefficient du log du prix N-1 : {m.coefs_['log_prix_n1'] / echelle_n1:.3f}")

H7 – Elastic Net – sous-bassin + bien + territoire + prix N-1 | fenêtre 2023 : 0.2 min
H7 – Elastic Net – sous-bassin + bien + territoire + prix N-1 | fenêtre 2024 : 0.3 min
H7 – Elastic Net – sous-bassin + bien + territoire + prix N-1 (25 variables) 
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       718.0             732.6                    2.0                            60.0          B
      maison         2023       756.0             774.2                    2.3                            61.5          B
 appartement         2023       217.8             166.2                  -31.1                            32.9          C
      maison         2023       155.2             111.7                  -38.9                            31.5          C
 appartement         2024       731.1             744.9                    1.8                            63.5          B
      maison    

**Résultats :**
- Avec le prix N-1, l'Elastic Net fait un peu mieux que la naïve pour chaque vente (évaluation B) : gain de 1,8 à 2,6 %, et il fait mieux dans 60 à 64 % des sous-bassins. Le gain reste sous le seuil de 3 % de la règle de décision.
- Pour la médiane d'un sous-bassin (évaluation C), il perd en 2023 (−31 % pour les appartements, −39 % pour les maisons) et pour les appartements en 2024 (−22 %), mais il bat la naïve pour les maisons en 2024 : 121 contre 129 €/m² (+6,5 %).
- Le coefficient du log du prix N-1 vaut 0,84 (fenêtre 2023) puis 0,79 (fenêtre 2024). Sous 1, il rapproche de la moyenne les prix des territoires chers et des territoires bon marché, comme le modèle pont (H4).
- Le réglage retient un `alpha` plus fort qu'en 4.1 (0,003 puis 0,01) : la pénalité sert davantage quand plusieurs variables (prix N-1, sous-bassin, territoire) disent le niveau des prix du lieu.

### 4.9 H7v3 : réseau Keras, H7v2a + prix N-1
H7v2a (sous-bassin, bien, territoire) avec le prix N-1. Question : le niveau récent du marché local corrige-t-il l'erreur de niveau des versions du « où » ?

In [24]:
H7V3 = lancer("H7v3 – Keras – sous-bassin + bien + territoire + prix N-1", "keras",
              ["sous_bassin_de_vie"] + BLOC_BIEN + BLOC_TERRITOIRE + ["log_prix_n1"])
biais([REFERENCE_OU, H7V1, H7V2A, H7V2B, REFERENCE_QUAND, H7V3])

H7v3 – Keras – sous-bassin + bien + territoire + prix N-1 | fenêtre 2023 : 2.7 min
H7v3 – Keras – sous-bassin + bien + territoire + prix N-1 | fenêtre 2024 : 3.6 min
H7v3 – Keras – sous-bassin + bien + territoire + prix N-1 (25 variables) {'dropout': 0.1, 'learning_rate': 0.0003, 'dim_embedding': 4, 'perte': 'mae', 'couches': (128, 64, 32), 'reference': 'log_prix_n1'}
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       698.6             732.6                    4.6                            63.1          B
      maison         2023       725.4             774.2                    6.3                            70.2          B
 appartement         2023       211.9             166.2                  -27.5                            40.8          C
      maison         2023       195.1             111.7                  -74.6                            29.4          C
 appartement       

année jugée                                                      2023  2024
modèle                                             type de bien            
H7 – Elastic Net – sous-bassin + bien              appartement   -8.9  -5.6
                                                   maison       -11.4  -6.8
H7 – Elastic Net – sous-bassin + bien + territo... appartement    4.3   5.1
                                                   maison         3.5   5.9
H7v1 – Keras – sous-bassin + bien                  appartement   -9.8  -6.5
                                                   maison        -7.5  -2.4
H7v2a – Keras – sous-bassin + bien + territoire    appartement  -10.4  -8.1
                                                   maison        -7.6  -2.8
H7v2b – Keras – bien + territoire, sans embedding  appartement   -9.5  -7.6
                                                   maison        -7.3  -2.9
H7v3 – Keras – sous-bassin + bien + territoire ... appartement    9.4   9.6
                                                   maison        10.9  13.7

**Résultats :**

- Avec le prix N-1, le réseau bat la naïve pour chaque vente (évaluation B) dans les quatre cas : gain de 4,6 % (appartements) et 6,3 % (maisons) en 2023, de 5,0 % et 4,8 % en 2024, toujours au-dessus du seuil de 3 %. Il fait mieux que la naïve dans 63 à 70 % des sous-bassins. Les tests de la section 5.2 diront si ces années sont réussies (IC et rang-bisériale).
- En évaluation C, il perd encore contre la naïve (de −28 à −97 %). Dans la médiane d'un sous-bassin, l'effet du bien disparaît : il ne reste qu'à prévoir l'évolution du marché.
- À informations égales, le réseau fait mieux que l'Elastic Net pour chaque vente (évaluation B : de 16 à 31 €/m² de moins selon l'année et le type de bien). Pour la médiane d'un sous-bassin (évaluation C), l'Elastic Net fait mieux, sauf pour les appartements en 2023 : 155 contre 195 €/m² pour les maisons en 2023, 179 contre 235 pour les appartements et 121 contre 254 pour les maisons en 2024.
- Le biais explique cet écart : les deux modèles surestiment les prix, l'Elastic Net de 3,5 à 5,9 %, le réseau de 9 à 14 %. Tous deux prolongent la hausse apprise sur 2017-2022 (même constat que pour le modèle pont, H4), le réseau davantage. Hypothèse : dans l'évaluation B, cette erreur de niveau se perd dans l'écart propre à chaque vente ; dans la médiane (évaluation C), elle reste entière.
- Comme attendu, le modèle par vente n'anticipe pas mieux le retournement : le biais du réseau augmente en 2024 (de +9,4 à +9,6 % pour les appartements, de +10,9 à +13,7 % pour les maisons), l'année où les prix baissent. Il sépare l'effet du bien de celui du marché, mais ne prévoit pas mieux le marché.

### 4.10 Le « quand » du niveau 2 : préparation des indicateurs
Les hypothèses H5 (volume et élan) et H6 (capacité d'achat) sont reprises sur les ventes, avec le réseau. Chaque version part de H7v3 (sous-bassin, bien, territoire, prix N-1) et ajoute un bloc d'indicateurs connus avant l'année jugée.
- **Volume et élan** (préparation P9), avec les indicateurs retenus par l'analyse préalable de H5 : nombre de ventes N-1, rotation du parc du même type (ventes N-1 rapportées au parc du même type) et élan (évolution du prix médian de N-2 à N-1). L'évolution du nombre de ventes, qui ne sépare pas les territoires en hausse, stables et en baisse, et la rotation du parc total, redondante, ne sont pas reprises. Quand le territoire avait moins de 20 ventes en N-2, l'élan est inconnu : il est mis à 0, et une indicatrice le signale. Aucune vente n'est retirée : toutes les versions sont jugées sur les mêmes ventes.
- **Capacité d'achat** (préparation P7) : taux d'effort, mensualité d'un achat type au prix N-1, au taux de décembre N-1, rapportée au revenu local.
- Le nombre de ventes, la rotation et le taux d'effort, très asymétriques, passent en log.

Les indicateurs sont joints aux ventes et à la table (pour le panier fixe), sur l'année, le sous-bassin et le type de bien.

In [25]:
CLES_INDICATEURS = ["annee", "sous_bassin_de_vie", "type_bien"]


def cles_de(t):
    """Année, sous-bassin et type de bien de chaque ligne, dans l'ordre des lignes (ventes : colonnes ; table : index)."""
    if "sous_bassin_de_vie" in t.columns:
        return t[CLES_INDICATEURS].reset_index(drop=True)
    return t.index.to_frame(index=False)[CLES_INDICATEURS]


def ajouter(t, valeurs, colonnes):
    """Ajoute à t (ventes ou table) les colonnes de `valeurs`, jointes sur l'année, le sous-bassin et le type de bien."""
    t = t.copy()
    k = cles_de(t).merge(valeurs[CLES_INDICATEURS + colonnes], on=CLES_INDICATEURS, how="left", validate="many_to_one")
    t[colonnes] = k[colonnes].to_numpy()
    return t


# Volume et élan (P9) : indicateurs retenus par l'analyse préalable de H5. L'élan est vide quand le territoire avait
# moins de 20 ventes en N-2 (appartements surtout) : vide -> 0 et indicatrice evol_n2_inconnue = 1, aucune vente retirée
P9 = pd.read_parquet(commun.DATA_OUT / "fe_volume_elan_sbv.parquet")
COLONNES_P9 = ["evol_prix_n2_n1", "rotation_parc_type"]
# Capacité d'achat (P7)
P7 = pd.read_parquet(commun.DATA_OUT / "fe_capacite_achat_sbv.parquet")


def preparer_quand_niveau2(t):
    """Indicateurs P9 et P7 ajoutés à t ; asymétriques en log ; élan vide remplacé par 0 avec une indicatrice."""
    t = ajouter(ajouter(t, P9, COLONNES_P9), P7, ["taux_effort"])
    t["evol_n2_inconnue"] = t["evol_prix_n2_n1"].isna().astype(int)
    t["evol_prix_n2_n1"] = t["evol_prix_n2_n1"].fillna(0)
    for v in ["nb_ventes_n1", "rotation_parc_type", "taux_effort"]:
        assert (t[v] > 0).all(), f"{v} : valeur vide, nulle ou négative"
        t["log_" + v] = np.log(t[v])
    return t


table = preparer_quand_niveau2(table)          # pour le panier fixe (évaluation C)
ventes = preparer_quand_niveau2(ventes)
FENETRES = fenetres(ventes, annee_debut=ANNEE_DEBUT_H7)

BLOC_H5 = ["log_nb_ventes_n1", "log_rotation_parc_type", "evol_prix_n2_n1", "evol_n2_inconnue"]
BLOC_H6 = ["log_taux_effort"]
BASE_V3 = ["sous_bassin_de_vie"] + BLOC_BIEN + BLOC_TERRITOIRE + ["log_prix_n1"]
assert ventes[BLOC_H5 + BLOC_H6].notna().all().all() and table[BLOC_H5 + BLOC_H6].notna().all().all()
print("Ventes 2017-2024 dont l'élan est inconnu (%) :")
print(ventes[ventes["annee"] >= ANNEE_DEBUT_H7].groupby("type_bien")["evol_n2_inconnue"].mean().mul(100).round(2).to_string())

Ventes 2017-2024 dont l'élan est inconnu (%) :
type_bien
appartement    0.27
maison         0.01


**Constat :** l'élan est inconnu pour 0,27 % des ventes d'appartements et 0,01 % des ventes de maisons (2017-2024) : l'indicatrice concerne très peu de ventes.

### 4.11 H7vH5 : réseau Keras, H7v3 + volume et élan
Hypothèse H5 sur les ventes : le volume de ventes et l'élan des prix de l'année précédente annoncent-ils le sens de l'évolution ?

In [26]:
H7VH5 = lancer("H7vH5 – Keras – H7v3 + volume et élan", "keras", BASE_V3 + BLOC_H5)

H7vH5 – Keras – H7v3 + volume et élan | fenêtre 2023 : 2.7 min
H7vH5 – Keras – H7v3 + volume et élan | fenêtre 2024 : 8.2 min
H7vH5 – Keras – H7v3 + volume et élan (29 variables) {'dropout': 0.1, 'learning_rate': 0.0003, 'dim_embedding': 4, 'perte': 'mae', 'couches': (128, 64, 32), 'reference': 'log_prix_n1'}
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       695.2             732.6                    5.1                            65.7          B
      maison         2023       725.4             774.2                    6.3                            69.9          B
 appartement         2023       193.1             166.2                  -16.2                            43.3          C
      maison         2023       201.3             111.7                  -80.2                            29.2          C
 appartement         2024       706.9             744.9                    5.1 

**Résultats :** le volume et l'élan changent peu H7v3 pour chaque vente. En évaluation B, l'erreur de H7vH5 est égale ou un peu inférieure à celle de H7v3 (de 0 à −7 €/m², moins de 1 %), et le gain sur la naïve reste au-dessus du seuil de 3 % dans les quatre cas (de 5,1 à 6,3 %). En évaluation C, l'erreur baisse dans trois cas sur quatre (−19 €/m² pour les appartements en 2023 et en 2024, −41 €/m² pour les maisons en 2024) et monte un peu pour les maisons en 2023 (+6 €/m²). Le biais explique ce gain : en 2024, il passe de +9,6 à +7,0 % pour les appartements et de +13,7 à +10,6 % pour les maisons ; en 2023, il ne change presque pas. H7vH5 perd encore contre la naïve sur la médiane d'un sous-bassin (de −16 à −80 %), car il surestime toujours les prix de 7 à 11 %. Les sections 5.1 (contre la naïve) et 5.2 (contre H7v3) disent si ces écarts comptent.

### 4.12 H7vH6 : réseau Keras, H7v3 + capacité d'achat
Hypothèse H6 sur les ventes : le taux d'effort d'un achat type, qui suit la remontée des taux des crédits, annonce-t-il le retournement ?

In [27]:
H7VH6 = lancer("H7vH6 – Keras – H7v3 + capacité d'achat", "keras", BASE_V3 + BLOC_H6)

H7vH6 – Keras – H7v3 + capacité d'achat | fenêtre 2023 : 4.3 min
H7vH6 – Keras – H7v3 + capacité d'achat | fenêtre 2024 : 5.6 min
H7vH6 – Keras – H7v3 + capacité d'achat (26 variables) {'dropout': 0.1, 'learning_rate': 0.0003, 'dim_embedding': 4, 'perte': 'mae', 'couches': (128, 64, 32), 'reference': 'log_prix_n1'}
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       691.1             732.6                    5.7                            67.3          B
      maison         2023       720.0             774.2                    7.0                            71.8          B
 appartement         2023       199.0             166.2                  -19.7                            41.5          C
      maison         2023       159.0             111.7                  -42.3                            36.2          C
 appartement         2024       706.7             744.9                  

**Résultats :** la capacité d'achat est la variable du « quand » qui apporte le plus. En évaluation B, l'erreur de H7vH6 baisse dans les quatre cas par rapport à H7v3 (de −1 à −13 €/m²), et le gain sur la naïve va de 5,1 à 7,0 %, au-dessus du seuil de 3 % ; le modèle fait mieux que la naïve dans 66 à 72 % des sous-bassins. En évaluation C, l'erreur baisse aussi dans les quatre cas, nettement pour les maisons (159 contre 195 €/m² en 2023, 170 contre 254 en 2024).

Le biais explique ce gain : il passe de +9,4 à +6,5 % (2023) et de +9,6 à +2,3 % (2024) pour les appartements, de +10,9 à +5,6 % et de +13,7 à +1,8 % pour les maisons. Le taux d'effort suit la remontée des taux des crédits en 2022-2023 : c'est la seule variable qui ramène le niveau prévu pour 2024 près du niveau réel.

H7vH6 perd pourtant encore contre la naïve sur la médiane d'un sous-bassin (de −20 à −42 %), et il fait moins bien que l'Elastic Net du « quand » en 2024 (203 contre 179 €/m² pour les appartements, 170 contre 121 pour les maisons). Hypothèse : en 2024, le biais moyen est presque corrigé (+2 %), mais l'erreur restante varie d'un sous-bassin à l'autre ; corriger le niveau moyen ne suffit pas à battre le prix N-1 de chaque sous-bassin. Les sections 5.1 et 5.2 disent si ces écarts comptent.

### 4.13 Les trimestres précédents : préparation
Les indicateurs annuels ne voient le retournement qu'avec un an de retard. Les dates des ventes permettent de regarder la fin de l'année N-1 : un ralentissement du prix ou une chute du volume au second semestre peut annoncer le retournement de l'année N.
- Un sous-bassin compte trop peu de ventes de l'échantillon par trimestre. Le calcul se fait au **département** (deux premiers caractères du code du sous-bassin) et par type de bien.
- Pour l'année N : prix médian (pondéré par `w_h`) et volume estimé (somme des `w_h`) des **trimestres 3 et 4 de N-1**, comparés aux mêmes trimestres de N-2 (évolution sur un an, en %). Comparer un trimestre au même trimestre de l'année d'avant retire la saisonnalité.
- Si un département compte moins de 30 ventes dans l'un des trimestres comparés, la valeur nationale du type de bien la remplace ; une indicatrice le signale.
- Seules des ventes de N-1 et N-2 servent : rien de l'année jugée.

In [28]:
from commun import mediane_ponderee

SEUIL_TRIMESTRE = 30          # ventes de l'échantillon par département, type de bien et trimestre
TRIMESTRES = [3, 4]           # les deux derniers trimestres de N-1


def stats_trimestre(v, lieu):
    """Prix médian pondéré (w_h), volume estimé (somme des w_h) et nombre de ventes, par lieu, année et trimestre."""
    g = v.groupby(lieu + ["anneemut", "trimestre"])
    return pd.DataFrame({"mediane": g.apply(lambda e: mediane_ponderee(e["prix_m2"], e["w_h"]), include_groups=False),
                         "volume": g["w_h"].sum(), "ventes": g.size()}).reset_index()


def glissements(s, lieu):
    """Pour l'année N : évolution du trimestre q de N-1 par rapport au trimestre q de N-2 (prix médian et volume, en %)."""
    sorties = []
    for q in TRIMESTRES:
        e = s[s["trimestre"] == q].drop(columns="trimestre")
        n1 = e.assign(annee=e["anneemut"] + 1).drop(columns="anneemut")     # trimestre q de N-1, rattaché à l'année N
        n2 = e.assign(annee=e["anneemut"] + 2).drop(columns="anneemut")     # trimestre q de N-2, rattaché à l'année N
        m = n1.merge(n2, on=lieu + ["annee"], suffixes=("_n1", "_n2"))
        sorties.append(pd.DataFrame({f"gliss_prix_t{q}": (m["mediane_n1"] / m["mediane_n2"] - 1).to_numpy() * 100,
                                     f"gliss_volume_t{q}": (m["volume_n1"] / m["volume_n2"] - 1).to_numpy() * 100,
                                     f"fiable_t{q}": (m[["ventes_n1", "ventes_n2"]].min(axis=1) >= SEUIL_TRIMESTRE).to_numpy()},
                                    index=pd.MultiIndex.from_frame(m[lieu + ["annee"]])))
    return pd.concat(sorties, axis=1).reset_index()


assert ventes_brutes["anneemut"].max() <= commun.ANNEES_JUGEES[-1], "aucune vente de 2025 ne doit servir"
vb = ventes_brutes.assign(departement=ventes_brutes["sous_bassin_de_vie"].str[:2],
                          trimestre=(ventes_brutes["moismut"] - 1) // 3 + 1)
G_DEP = glissements(stats_trimestre(vb, ["departement", "type_bien"]), ["departement", "type_bien"])
G_NAT = glissements(stats_trimestre(vb, ["type_bien"]), ["type_bien"])
BLOC_TRIMESTRES = [f"gliss_{x}_t{q}" for q in TRIMESTRES for x in ("prix", "volume")]


def preparer_trimestres(t):
    """Glissements des trimestres 3 et 4 de N-1 du département ; valeur nationale si moins de 30 ventes."""
    t = t.copy()
    k = cles_de(t).assign(departement=lambda d: d["sous_bassin_de_vie"].str[:2])
    k = k.merge(G_DEP, on=["departement", "type_bien", "annee"], how="left", validate="many_to_one")
    k = k.merge(G_NAT, on=["type_bien", "annee"], how="left", validate="many_to_one", suffixes=("", "_nat"))
    for q in TRIMESTRES:
        national = ~k[f"fiable_t{q}"].eq(True)        # département absent ou trop petit : valeur nationale
        for x in ("prix", "volume"):
            c = f"gliss_{x}_t{q}"
            k.loc[national, c] = k.loc[national, c + "_nat"]
    t[BLOC_TRIMESTRES] = k[BLOC_TRIMESTRES].to_numpy(dtype=float)
    t["trimestre_national"] = (~k[[f"fiable_t{q}" for q in TRIMESTRES]].eq(True).all(axis=1)).astype(int).to_numpy()
    return t


table = preparer_trimestres(table)
ventes = preparer_trimestres(ventes)
FENETRES = fenetres(ventes, annee_debut=ANNEE_DEBUT_H7)
assert ventes[BLOC_TRIMESTRES].notna().all().all() and table[BLOC_TRIMESTRES].notna().all().all()

print("Ventes avec au moins une valeur nationale (département trop petit), en % :")
print(ventes.groupby("annee")["trimestre_national"].mean().mul(100).round(1).loc[ANNEE_DEBUT_H7:].to_string())
print("\nGlissements nationaux (%), par type de bien et année de prix :")
print(G_NAT[G_NAT["annee"].between(ANNEE_DEBUT_H7, commun.ANNEES_JUGEES[-1])]
      .set_index(["type_bien", "annee"])[BLOC_TRIMESTRES].round(1).to_string())

Ventes avec au moins une valeur nationale (département trop petit), en % :
annee
2017    3.9
2018    4.4
2019    3.8
2020    3.6
2021    3.2
2022    3.2
2023    3.1
2024    4.3

Glissements nationaux (%), par type de bien et année de prix :
                   gliss_prix_t3  gliss_volume_t3  gliss_prix_t4  gliss_volume_t4
type_bien   annee                                                                
appartement 2017            -2.2              7.7            0.1              0.1
            2018             3.2              5.6            2.4              8.1
            2019             2.4              3.7           -0.7              6.7
            2020             0.6             10.8            3.1             10.8
            2021             2.8            -13.9            3.4             11.7
            2022             7.1             31.2            7.4             -6.9
            2023             6.8             -3.5            6.2             -3.8
            2024     

**Constat :**
- De 3,1 à 4,4 % des ventes, selon l'année, reçoivent au moins une valeur nationale (département trop petit pour un trimestre).
- Pour les prix 2023 (fin 2022), les prix des trimestres 3 et 4 montaient encore de 6 à 9 % sur un an ; seul le volume des maisons reculait (−11 %), celui des appartements à peine (−4 %). Le signal précurseur du retournement de 2023 est donc faible.
- Pour les prix 2024 (fin 2023), le signal est net : volume en baisse de 20 à 27 %, prix du 4e trimestre déjà en baisse (−3 % pour les appartements, −6 % pour les maisons).
- Dans les années d'apprentissage, le volume a déjà reculé sans baisse des prix ensuite : au 3e trimestre 2020 (prix 2021), après le confinement, de −10 à −14 % ; au 4e trimestre 2021 (prix 2022), de −7 à −13 %. Un recul du volume n'annonce donc pas à lui seul une baisse des prix.

### 4.14 H7vT : réseau Keras, H7v3 + trimestres précédents
H7v3 avec, en plus, l'évolution du prix et du volume des trimestres 3 et 4 de l'année précédente. Question : la fin de l'année précédente annonce-t-elle le retournement mieux que les indicateurs annuels ?

In [29]:
H7VT = lancer("H7vT – Keras – H7v3 + trimestres précédents", "keras",
              BASE_V3 + BLOC_TRIMESTRES + ["trimestre_national"])
biais([H7V3, H7VH5, H7VH6, H7VT])

H7vT – Keras – H7v3 + trimestres précédents | fenêtre 2023 : 2.8 min
H7vT – Keras – H7v3 + trimestres précédents | fenêtre 2024 : 7.5 min
H7vT – Keras – H7v3 + trimestres précédents (30 variables) {'dropout': 0.1, 'learning_rate': 0.0003, 'dim_embedding': 4, 'perte': 'mae', 'couches': (128, 64, 32), 'reference': 'log_prix_n1'}
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       697.8             732.6                    4.8                            66.3          B
      maison         2023       723.4             774.2                    6.6                            71.2          B
 appartement         2023       212.7             166.2                  -28.0                            41.1          C
      maison         2023       187.9             111.7                  -68.2                            31.6          C
 appartement         2024       707.2             744.9      

année jugée                                                      2023  2024
modèle                                             type de bien            
H7v3 – Keras – sous-bassin + bien + territoire ... appartement    9.4   9.6
                                                   maison        10.9  13.7
H7vH5 – Keras – H7v3 + volume et élan              appartement    9.1   7.0
                                                   maison        11.3  10.6
H7vH6 – Keras – H7v3 + capacité d'achat            appartement    6.5   2.3
                                                   maison         5.6   1.8
H7vT – Keras – H7v3 + trimestres précédents        appartement    9.1   6.1
                                                   maison         9.5  11.4

**Résultats :** les trimestres précédents réduisent un peu l'erreur de niveau de 2024 sans la corriger. En évaluation B, l'erreur de H7vT est celle de H7v3 à 1 à 3 €/m² près (gain sur la naïve de 4,8 à 6,6 %). En évaluation C, elle baisse nettement en 2024 (201 contre 235 €/m² pour les appartements, 233 contre 254 pour les maisons) et ne change presque pas en 2023. Le biais va dans le même sens : en 2024, il passe de +9,6 à +6,1 % pour les appartements et de +13,7 à +11,4 % pour les maisons ; en 2023, il ne bouge presque pas (de +9,4 à +9,1 %, de +10,9 à +9,5 %). Le réseau surestime donc encore les prix de 6 à 11 %, alors que le volume de fin 2023 reculait de 20 à 27 % et que les prix du 4e trimestre baissaient déjà. Hypothèses : le réseau n'a jamais vu, à l'apprentissage, une chute du volume de 25 % accompagnée d'une baisse des prix ; et il y a vu des baisses du volume (fin 2020, fin 2021) qui n'annonçaient pas de baisse des prix. La capacité d'achat corrige davantage le biais de 2024 (H7vH6 : +2,3 et +1,8 %). Les sections 5.1 et 5.2 disent si ces écarts comptent.

### 4.15 H7v4 : réseau Keras, le « quand » complet
H7v3 avec les trois blocs du « quand » réunis : volume et élan, capacité d'achat, trimestres précédents. Question : les indicateurs se complètent-ils ? Sur la table agrégée, l'élan aide surtout en 2023 (année où la hausse s'arrête) et le taux d'effort surtout en 2024 (année où le crédit renchérit).

In [30]:
H7V4 = lancer("H7v4 – Keras – H7v3 + volume, élan, capacité d'achat et trimestres", "keras",
              BASE_V3 + BLOC_H5 + BLOC_H6 + BLOC_TRIMESTRES + ["trimestre_national"])
biais([H7V3, H7VH5, H7VH6, H7VT, H7V4])

H7v4 – Keras – H7v3 + volume, élan, capacité d'achat et trimestres | fenêtre 2023 : 2.9 min
H7v4 – Keras – H7v3 + volume, élan, capacité d'achat et trimestres | fenêtre 2024 : 5.6 min
H7v4 – Keras – H7v3 + volume, élan, capacité d'achat et trimestres (35 variables) {'dropout': 0.1, 'learning_rate': 0.0003, 'dim_embedding': 4, 'perte': 'mae', 'couches': (128, 64, 32), 'reference': 'log_prix_n1'}
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       692.0             732.6                    5.6                            65.9          B
      maison         2023       722.6             774.2                    6.7                            71.1          B
 appartement         2023       199.5             166.2                  -20.0                            40.8          C
      maison         2023       184.6             111.7                  -65.3                            31.7    

année jugée                                                      2023  2024
modèle                                             type de bien            
H7v3 – Keras – sous-bassin + bien + territoire ... appartement    9.4   9.6
                                                   maison        10.9  13.7
H7v4 – Keras – H7v3 + volume, élan, capacité d'... appartement    8.3   1.3
                                                   maison         9.2   3.8
H7vH5 – Keras – H7v3 + volume et élan              appartement    9.1   7.0
                                                   maison        11.3  10.6
H7vH6 – Keras – H7v3 + capacité d'achat            appartement    6.5   2.3
                                                   maison         5.6   1.8
H7vT – Keras – H7v3 + trimestres précédents        appartement    9.1   6.1
                                                   maison         9.5  11.4

**Résultats :** réunis, les indicateurs du « quand » se complètent surtout en 2024. Par rapport à H7v3, l'erreur baisse dans les huit cas : de 3 à 19 €/m² en évaluation B (gain sur la naïve de 5,4 à 7,3 %, le meilleur de la famille avec H7vH6), de 10 à 97 €/m² en évaluation C, avec les plus fortes baisses en 2024 (194 contre 235 €/m² pour les appartements, 157 contre 254 pour les maisons). Le biais de 2024 tombe à +1,3 % pour les appartements et +3,8 % pour les maisons (contre +9,6 et +13,7 % pour H7v3), mais celui de 2023 reste élevé (+8,3 et +9,2 %).

Par rapport au meilleur bloc seul (H7vH6, capacité d'achat), le gain est faible : en évaluation B, les deux modèles sont à quelques €/m² près ; en évaluation C, H7v4 fait mieux en 2024 (194 contre 203 €/m² pour les appartements, 157 contre 170 pour les maisons), mais moins bien pour les maisons en 2023 (185 contre 159 €/m²). La capacité d'achat porte l'essentiel ; volume, élan et trimestres ajoutent un peu en 2024, l'année où le marché se retourne.

H7v4 perd encore contre la naïve sur la médiane d'un sous-bassin (de −20 à −65 %). Les sections 5.1 et 5.2 disent si ces écarts comptent.

## 5. Mesures et tests
Trois tableaux, du plus simple au plus exigeant :
1. **`res` : les erreurs**, déjà affichées après chaque modèle : MAE du modèle et de la naïve en €/m², gain sur la naïve, part des sous-bassins où le modèle gagne. C'est une description : un modèle peut avoir une MAE plus basse par hasard.
2. **`tst` : les tests contre la naïve.** Ils disent si l'écart avec la naïve est plus grand que le hasard.
   - **Écart de MAE avec la naïve** : erreur du modèle moins erreur de la naïve, en €/m² (négatif : le modèle fait mieux). En évaluation B, l'erreur d'un sous-bassin est la moyenne des erreurs de ses ventes, pondérée par `w_h`.
   - **IC 95 %** (bootstrap par bassin de vie) : si toute la fourchette est sous 0, le modèle fait vraiment mieux ; si elle contient 0, on ne peut pas conclure ; si elle est toute au-dessus de 0, il fait vraiment moins bien.
   - **p-value de Wilcoxon** : le modèle fait-il mieux (ou moins bien) dans la plupart des sous-bassins ? On la lit avec la taille d'effet.
   - **Rang-bisériale** (taille d'effet) : de −1 (le modèle fait mieux partout) à +1 (il fait moins bien partout).
3. **La règle de décision** : une année est réussie si l'IC est entièrement sous 0, la rang-bisériale négative et le gain d'au moins 3 %. Un modèle est retenu s'il réussit les deux années jugées.

Le bootstrap mesure le hasard dû aux territoires, pas celui de l'entraînement du réseau : deux entraînements avec des graines différentes donneraient des résultats un peu différents.

In [31]:
pd.set_option("display.max_rows", 200)      # tableaux affichés en entier
res = pd.DataFrame(resultats)
tst = pd.DataFrame(tests)
tst[["modèle", "type de bien", "année jugée", "écart de MAE avec la naïve (€/m²)",
     "IC 95 % bas (€/m²)", "IC 95 % haut (€/m²)", "p-value Wilcoxon", "rang-bisériale"]].round(3)

,modèle,type de bien,année jugée,écart de MAE avec la naïve (€/m²),IC 95 % bas (€/m²),IC 95 % haut (€/m²),p-value Wilcoxon,rang-bisériale
0,"H7 – Elastic Net – sous-bassin + bien (B, ventes)",appartement,2023,112.203,94.562,129.776,0.000,0.521
1,"H7 – Elastic Net – sous-bassin + bien (B, ventes)",maison,2023,86.792,70.337,105.833,0.000,0.545
2,"H7 – Elastic Net – sous-bassin + bien (C, pani...",appartement,2023,331.695,307.398,357.684,0.000,0.854
3,"H7 – Elastic Net – sous-bassin + bien (C, pani...",maison,2023,347.170,317.855,382.873,0.000,0.950
4,"H7 – Elastic Net – sous-bassin + bien (B, ventes)",appartement,2024,96.270,76.630,115.800,0.000,0.416
5,"H7 – Elastic Net – sous-bassin + bien (B, ventes)",maison,2024,48.177,33.439,66.175,0.000,0.316
6,"H7 – Elastic Net – sous-bassin + bien (C, pani...",appartement,2024,303.933,275.366,335.368,0.000,0.801
7,"H7 – Elastic Net – sous-bassin + bien (C, pani...",maison,2024,230.269,205.306,260.228,0.000,0.802
8,"H7v1 – Keras – sous-bassin + bien (B, ventes)",appartement,2023,70.161,44.493,98.511,0.000,0.273
9,"H7v1 – Keras – sous-bassin + bien (B, ventes)",maison,2023,17.392,7.886,27.383,0.583,0.015


**Comment lire le tableau des tests :** prenons l'Elastic Net du « où », appartements, 2023, évaluation B. L'écart de MAE avec la naïve vaut +112 €/m² : le modèle se trompe en moyenne de 112 €/m² de plus que la naïve par sous-bassin. L'IC 95 % va de +95 à +130 €/m², entièrement au-dessus de 0 : le modèle fait vraiment moins bien. La rang-bisériale vaut +0,52 : il fait moins bien dans une nette majorité de sous-bassins. À l'inverse, H7v4, maisons, 2024, évaluation B : l'écart vaut −54 €/m², l'IC va de −60 à −47 €/m², entièrement sous 0, et la rang-bisériale vaut −0,54 : le modèle fait vraiment mieux que la naïve, dans une nette majorité de sous-bassins.

### 5.1 Règle de décision contre la naïve

In [32]:
SEUIL_GAIN = 3      # % de gain sur la MAE de la comparaison, fixé avant les résultats


def regle_decision(res, tst, comparaison="naïve"):
    """Années réussies et décision, par modèle et type de bien."""
    cles = ["modèle", "type de bien", "année jugée"]
    regle = res[cles + [f"gain sur la {comparaison} (%)"]].merge(
        tst[cles + ["IC 95 % haut (€/m²)", "rang-bisériale"]], on=cles)
    regle["année réussie"] = ((regle["IC 95 % haut (€/m²)"] < 0) & (regle["rang-bisériale"] < 0)
                              & (regle[f"gain sur la {comparaison} (%)"] >= SEUIL_GAIN))
    decision = regle.groupby(["modèle", "type de bien"], sort=False)["année réussie"].agg(["sum", "count"])
    decision["retenu"] = decision["sum"] > decision["count"] / 2
    return decision.rename(columns={"sum": "années réussies", "count": "années jugées"})


regle_decision(res, tst)

années réussies  années jugées  retenu
modèle                                             type de bien                                        
H7 – Elastic Net – sous-bassin + bien (B, ventes)  appartement                 0              2   False
                                                   maison                      0              2   False
H7 – Elastic Net – sous-bassin + bien (C, panie... appartement                 0              2   False
                                                   maison                      0              2   False
H7v1 – Keras – sous-bassin + bien (B, ventes)      appartement                 0              2   False
                                                   maison                      0              2   False
H7v1 – Keras – sous-bassin + bien (C, panier fixe) appartement                 0              2   False
                                                   maison                      0              2   False
H7v1b – Keras – sous-bassin + bien + coordonnée... appartement                 0              2   False
                                                   maison                      0              2   False
H7v1b – Keras – sous-bassin + bien + coordonnée... appartement                 0              2   False
                                                   maison                      0              2   False
H7v2a – Keras – sous-bassin + bien + territoire... appartement                 0              2   False
                                                   maison                      1              2   False
H7v2a – Keras – sous-bassin + bien + territoire... appartement                 0              2   False
                                                   maison                      0              2   False
H7v2b – Keras – bien + territoire, sans embeddi... appartement                 0              2   False
                                                   maison                      0              2   False
H7v2b – Keras – bien + territoire, sans embeddi... appartement                 0              2   False
                                                   maison                      0              2   False
H7 – Elastic Net – sous-bassin + bien + territo... appartement                 0              2   False
                                                   maison                      0              2   False
H7 – Elastic Net – sous-bassin + bien + territo... appartement                 0              2   False
                                                   maison                      1              2   False
H7v3 – Keras – sous-bassin + bien + territoire ... appartement                 2              2    True
                                                   maison                      2              2    True
H7v3 – Keras – sous-bassin + bien + territoire ... appartement                 0              2   False
                                                   maison                      0              2   False
H7vH5 – Keras – H7v3 + volume et élan (B, ventes)  appartement                 2              2    True
                                                   maison                      2              2    True
H7vH5 – Keras – H7v3 + volume et élan (C, panie... appartement                 0              2   False
                                                   maison                      0              2   False
H7vH6 – Keras – H7v3 + capacité d'achat (B, ven... appartement                 2              2    True
                                                   maison                      2              2    True
H7vH6 – Keras – H7v3 + capacité d'achat (C, pan... appartement                 0              2   False
                                                   maison                      0              2   False
H7vT – Keras – H7v3 + trimestres précédents (B,... appartement                 2              2    True
         

**Résultats contre la naïve :**

- **Modèles du « où »** (Elastic Net, H7v1, H7v1b, H7v2a, H7v2b) : aucun n'est retenu. Une seule année réussie, H7v2a pour les maisons en 2024 en évaluation B (gain de 3,1 %, IC 95 % de −30 à −15 €/m², rang-bisériale −0,27) ; H7v1 et H7v1b battent aussi la naïve pour les maisons en 2024, mais sous le seuil de 3 % (1,6 et 2,8 %).
- **Elastic Net du « quand »** : en évaluation B, il bat la naïve dans les quatre cas (IC entièrement sous 0), mais son gain (de 1,8 à 2,6 %) reste sous le seuil de 3 % ; en évaluation C, il réussit une année, les maisons en 2024 (gain de 6,5 %, IC de −15 à −2 €/m²). Non retenu.
- **Réseaux avec le prix N-1** (H7v3, H7vH5, H7vH6, H7vT, H7v4) : retenus en évaluation B pour les deux types de bien, les deux années. Les gains vont de 4,6 à 7,3 %, les IC sont entièrement sous 0 et les rangs-bisériales vont de −0,34 à −0,54. Les meilleurs sont H7vH6 et H7v4 (de −38 à −54 €/m² d'écart avec la naïve).
- **En évaluation C**, aucun réseau ne réussit une seule année : tous les IC sont entièrement au-dessus de 0.
- Le modèle par vente bat donc la naïve pour prévoir le prix d'un bien vendu, à condition de connaître le prix N-1, mais pas pour prévoir le prix médian d'un territoire. Le seul cas où un modèle par vente bat la naïve sur la médiane d'un sous-bassin est l'Elastic Net du « quand », pour les maisons en 2024, une seule année.

### 5.2 Les versions entre elles
Chaque comparaison répond à une question du plan ; on applique les mêmes mesures, les mêmes tests et la même règle de décision, en remplaçant la naïve par la prévision du modèle de comparaison :
- en évaluation B, la prévision de la référence remplace le prix N-1 de chaque vente ;
- en évaluation C, la médiane des prévisions de la référence remplace le prix N-1 du sous-bassin.

| Comparaison | Question |
|---|---|
| H7v1 contre l'Elastic Net du « où » | À informations égales, le réseau fait-il mieux qu'un Elastic Net ? |
| H7v1b contre H7v1 | La localisation à l'intérieur du sous-bassin apporte-t-elle quelque chose ? |
| H7v2a contre H7v1 | Le territoire ajoute-t-il quelque chose à l'identité du sous-bassin ? |
| H7v2b contre H7v1 | Le territoire peut-il remplacer l'identité du sous-bassin ? |
| H7v3 contre l'Elastic Net du « quand » | Même question que la première, avec le prix N-1 |
| H7v3 contre H7v2a | Qu'apporte le prix N-1 ? |
| H7vH5 contre H7v3 | Le volume et l'élan apportent-ils quelque chose au prix N-1 (H5) ? |
| H7vH6 contre H7v3 | La capacité d'achat apporte-t-elle quelque chose au prix N-1 (H6) ? |
| H7vT contre H7v3 | Les trimestres précédents apportent-ils quelque chose au prix N-1 ? |
| H7v4 contre H7v3 | Les indicateurs du « quand » réunis apportent-ils quelque chose au prix N-1 ? |

Lecture : un gain positif et un IC entièrement sous 0 veulent dire que le modèle fait vraiment mieux que sa comparaison.

In [33]:
from commun import CLES, mediane_ponderee, evaluer, tester


def medianes_panier(panier, prevision):
    """Médiane pondérée (w_h) des prévisions du panier fixe, par sous-bassin (comme evaluer_panier)."""
    d = pd.DataFrame({"p": prevision.loc[panier.index], "w": panier["w_h"], **{c: panier[c] for c in CLES}})
    return d.groupby(table.index.names).apply(lambda g: mediane_ponderee(g["p"], g["w"]), include_groups=False)


COMPARAISONS = [  # (libellé, modèle, modèle de comparaison)
    ("H7v1 contre l'Elastic Net du « où »", H7V1, REFERENCE_OU),
    ("H7v1b contre H7v1 (coordonnées)", H7V1B, H7V1),
    ("H7v2a contre H7v1 (territoire en plus)", H7V2A, H7V1),
    ("H7v2b contre H7v1 (territoire à la place du sous-bassin)", H7V2B, H7V1),
    ("H7v3 contre l'Elastic Net du « quand »", H7V3, REFERENCE_QUAND),
    ("H7v3 contre H7v2a (prix N-1 en plus)", H7V3, H7V2A),
    ("H7vH5 contre H7v3 (volume et élan en plus)", H7VH5, H7V3),
    ("H7vH6 contre H7v3 (capacité d'achat en plus)", H7VH6, H7V3),
    ("H7vT contre H7v3 (trimestres précédents en plus)", H7VT, H7V3),
    ("H7v4 contre H7v3 (tout le « quand » en plus)", H7V4, H7V3),
]

resultats_ref, tests_ref = [], []
for libelle, nom, nom_ref in COMPARAISONS:
    for annee, (app, ev) in FENETRES.items():
        p, p_panier = previsions[(nom, annee)]
        r, r_panier = previsions[(nom_ref, annee)]
        # B : la prévision de la référence remplace la naïve (prix_n1) pour chaque vente
        ev_ref = ev.copy()
        ev_ref["prix_n1"] = r
        evaluer_ventes(libelle, ev_ref, p, resultats_ref)
        tester_ventes(libelle, ev_ref, p, tests_ref)
        # C : la médiane des prévisions de la référence remplace le prix N-1 du sous-bassin
        panier = panier_fixe(ventes_brutes, table, annee)
        med, med_ref = medianes_panier(panier, p_panier), medianes_panier(panier, r_panier)
        evaluation = table.loc[table.index.intersection(med.index)].copy()
        evaluation["prix_n1"] = med_ref.loc[evaluation.index]
        evaluer(f"{libelle} (C, panier fixe)", evaluation, med.loc[evaluation.index], resultats_ref)
        tester(f"{libelle} (C, panier fixe)", evaluation, med.loc[evaluation.index], tests_ref)

# Mêmes tableaux que contre la naïve, avec « référence » à la place de « naïve »
res_ref = pd.DataFrame(resultats_ref).rename(columns=lambda c: c.replace("naïve", "référence"))
tst_ref = pd.DataFrame(tests_ref).rename(columns=lambda c: c.replace("naïve", "référence"))
display(res_ref[["modèle", "type de bien", "année jugée", "MAE (€/m²)", "MAE référence (€/m²)",
                 "gain sur la référence (%)", "bassins où le modèle gagne (%)"]].round(2))
display(tst_ref[["modèle", "type de bien", "année jugée", "écart de MAE avec la référence (€/m²)",
                 "IC 95 % bas (€/m²)", "IC 95 % haut (€/m²)", "p-value Wilcoxon", "rang-bisériale"]].round(3))

,modèle,type de bien,année jugée,MAE (€/m²),MAE référence (€/m²),gain sur la référence (%),bassins où le modèle gagne (%)
0,"H7v1 contre l'Elastic Net du « où » (B, ventes)",appartement,2023,802.79,844.83,4.98,66.16
1,"H7v1 contre l'Elastic Net du « où » (B, ventes)",maison,2023,791.57,860.97,8.06,78.43
2,"H7v1 contre l'Elastic Net du « où » (C, panier...",appartement,2023,454.59,497.92,8.70,64.88
3,"H7v1 contre l'Elastic Net du « où » (C, panier...",maison,2023,342.51,458.88,25.36,78.37
4,"H7v1 contre l'Elastic Net du « où » (B, ventes)",appartement,2024,803.89,841.15,4.43,63.91
5,"H7v1 contre l'Elastic Net du « où » (B, ventes)",maison,2024,728.99,788.78,7.58,74.39
6,"H7v1 contre l'Elastic Net du « où » (C, panier...",appartement,2024,417.41,451.03,7.45,63.39
7,"H7v1 contre l'Elastic Net du « où » (C, panier...",maison,2024,256.05,359.46,28.77,74.49
8,"H7v1b contre H7v1 (coordonnées) (B, ventes)",appartement,2023,792.63,802.79,1.27,54.25
9,"H7v1b contre H7v1 (coordonnées) (B, ventes)",maison,2023,790.49,791.57,0.14,51.91


,modèle,type de bien,année jugée,écart de MAE avec la référence (€/m²),IC 95 % bas (€/m²),IC 95 % haut (€/m²),p-value Wilcoxon,rang-bisériale
0,"H7v1 contre l'Elastic Net du « où » (B, ventes)",appartement,2023,-42.042,-61.814,-16.476,0.000,-0.368
1,"H7v1 contre l'Elastic Net du « où » (B, ventes)",maison,2023,-69.400,-82.971,-59.001,0.000,-0.675
2,"H7v1 contre l'Elastic Net du « où » (C, panier...",appartement,2023,-43.330,-86.691,19.926,0.000,-0.341
3,"H7v1 contre l'Elastic Net du « où » (C, panier...",maison,2023,-116.370,-132.456,-102.135,0.000,-0.684
4,"H7v1 contre l'Elastic Net du « où » (B, ventes)",appartement,2024,-37.257,-49.570,-24.697,0.000,-0.342
5,"H7v1 contre l'Elastic Net du « où » (B, ventes)",maison,2024,-59.788,-72.306,-49.195,0.000,-0.625
6,"H7v1 contre l'Elastic Net du « où » (C, panier...",appartement,2024,-33.621,-54.760,-12.585,0.000,-0.287
7,"H7v1 contre l'Elastic Net du « où » (C, panier...",maison,2024,-103.406,-121.038,-90.231,0.000,-0.630
8,"H7v1b contre H7v1 (coordonnées) (B, ventes)",appartement,2023,-10.156,-21.605,-1.312,0.014,-0.098
9,"H7v1b contre H7v1 (coordonnées) (B, ventes)",maison,2023,-1.081,-4.149,1.961,0.181,-0.037


#### Règle de décision contre la comparaison

In [34]:
regle_decision(res_ref, tst_ref, comparaison="référence")

années réussies  années jugées  retenu
modèle                                             type de bien                                        
H7v1 contre l'Elastic Net du « où » (B, ventes)    appartement                 2              2    True
                                                   maison                      2              2    True
H7v1 contre l'Elastic Net du « où » (C, panier ... appartement                 1              2   False
                                                   maison                      2              2    True
H7v1b contre H7v1 (coordonnées) (B, ventes)        appartement                 0              2   False
                                                   maison                      0              2   False
H7v1b contre H7v1 (coordonnées) (C, panier fixe)   appartement                 0              2   False
                                                   maison                      0              2   False
H7v2a contre H7v1 (territoire en plus) (B, ventes) appartement                 0              2   False
                                                   maison                      0              2   False
H7v2a contre H7v1 (territoire en plus) (C, pani... appartement                 0              2   False
                                                   maison                      0              2   False
H7v2b contre H7v1 (territoire à la place du sou... appartement                 0              2   False
                                                   maison                      0              2   False
H7v2b contre H7v1 (territoire à la place du sou... appartement                 0              2   False
                                                   maison                      0              2   False
H7v3 contre l'Elastic Net du « quand » (B, ventes) appartement                 1              2   False
                                                   maison                      1              2   False
H7v3 contre l'Elastic Net du « quand » (C, pani... appartement                 0              2   False
                                                   maison                      0              2   False
H7v3 contre H7v2a (prix N-1 en plus) (B, ventes)   appartement                 2              2    True
                                                   maison                      1              2   False
H7v3 contre H7v2a (prix N-1 en plus) (C, panier... appartement                 2              2    True
                                                   maison                      1              2   False
H7vH5 contre H7v3 (volume et élan en plus) (B, ... appartement                 0              2   False
                                                   maison                      0              2   False
H7vH5 contre H7v3 (volume et élan en plus) (C, ... appartement                 2              2    True
                                                   maison                      1              2   False
H7vH6 contre H7v3 (capacité d'achat en plus) (B... appartement                 0              2   False
                                                   maison                      0              2   False
H7vH6 contre H7v3 (capacité d'achat en plus) (C... appartement                 2              2    True
                                                   maison                      2              2    True
H7vT contre H7v3 (trimestres précédents en plus... appartement                 0              2   False
                                                   maison                      0              2   False
H7vT contre H7v3 (trimestres précédents en plus... appartement                 1              2   False
                                                   maison                      2              2    True
H7v4 contre H7v3 (tout le « quand » en plus) (B... appartement                 0              2   False
         

**Comment lire :** prenons « H7v3 contre H7v2a », maisons, 2023, évaluation C. L'écart de MAE avec la référence vaut −154 €/m² : avec le prix N-1, l'erreur sur la médiane d'un sous-bassin baisse de 154 €/m² par rapport au même réseau sans prix N-1 (gain de 44 %). L'IC 95 % va de −181 à −131 €/m², entièrement sous 0, et la rang-bisériale vaut −0,45 : H7v3 fait mieux dans une nette majorité de sous-bassins.

**Résultats :**

- **Réseau contre Elastic Net, sans prix N-1 (H7v1 contre l'Elastic Net du « où »)** : le réseau est retenu en évaluation B pour les deux types de bien (gain de 4,4 à 8,1 %, IC entièrement sous 0, rang-bisériale de −0,34 à −0,68), et en évaluation C pour les maisons (gain de 25 et 29 %). Pour les appartements en évaluation C, il ne réussit que 2024 (l'IC de 2023 contient 0). À informations égales, le réseau décrit nettement mieux le bien et le lieu que l'Elastic Net.
- **Les coordonnées (H7v1b contre H7v1)** : gain de 0,1 à 1,3 % en évaluation B, de −0,3 à 2,8 % en évaluation C ; l'IC est sous 0 dans quelques cas (appartements 2023 et maisons 2024 en B), mais le gain reste sous le seuil de 3 %. Non retenu : la localisation à l'intérieur du sous-bassin est presque redondante avec le sous-bassin.
- **Le territoire en plus du sous-bassin (H7v2a contre H7v1)** : gain de 0,4 à 1,5 % en évaluation B (IC sous 0 seulement pour les maisons) ; en évaluation C, erreur un peu plus grande en 2023 (IC au-dessus de 0 pour les maisons), un peu plus petite en 2024 mais avec un IC qui contient 0. Non retenu.
- **Le territoire à la place du sous-bassin (H7v2b contre H7v1)** : erreur plus grande partout (de 1,5 à 2,4 % en évaluation B, de 6 à 18 % en évaluation C), IC au-dessus de 0 dans sept cas sur huit. Les 11 variables de territoire ne remplacent pas l'identité du sous-bassin.
- **Le prix N-1 (H7v3 contre H7v2a)** : retenu pour les appartements dans les deux évaluations (gain de 11 et 13 % en B, de 41 et 54 % en C). Pour les maisons, il réussit 2023 (8 % en B, 44 % en C) mais pas 2024 : le gain tombe à 1,8 % en B (sous le seuil) et l'IC de l'évaluation C contient 0.
- **Réseau contre Elastic Net, avec le prix N-1 (H7v3 contre l'Elastic Net du « quand »)** : en évaluation B, le réseau fait mieux dans les quatre cas (IC entièrement sous 0, rang-bisériale de −0,20 à −0,38), mais le gain ne dépasse 3 % qu'une année par type de bien (appartements 2024 : 3,2 % ; maisons 2023 : 4,1 %) : non retenu. En évaluation C, l'Elastic Net fait nettement mieux dans trois cas (écart de +40 à +133 €/m², IC entièrement au-dessus de 0) ; pour les appartements en 2023, l'IC contient 0.
- **Les indicateurs du « quand » du niveau 2 (H7vH5, H7vH6, H7vT et H7v4 contre H7v3)** : en évaluation B, aucun n'est retenu ; les gains vont de 0 à 2,6 %, sous le seuil, même quand l'IC est sous 0 (par exemple H7v4, maisons, 2024 : −19 €/m², IC de −24 à −14). En évaluation C, ils améliorent la médiane d'un sous-bassin : H7vH6 (capacité d'achat) et H7v4 (tout le « quand ») sont retenus pour les deux types de bien (écart de −10 à −97 €/m², rang-bisériale de −0,13 à −0,57), H7vT pour les maisons, H7vH5 pour les appartements. Le gain est le plus fort en 2024 (H7v4 : −42 €/m² pour les appartements, −97 pour les maisons), en accord avec la baisse du biais (sections 4.11 à 4.15).
- **Lecture d'ensemble** : les indicateurs du « quand » corrigent surtout le niveau des prix d'un sous-bassin (évaluation C), presque pas l'erreur sur chaque vente (évaluation B), qui est dominée par l'écart propre à chaque bien. Ils ne suffisent pourtant pas à battre la naïve en évaluation C (section 5.1).
- **Réserve** : le bootstrap mesure le hasard dû aux territoires, pas celui de l'entraînement du réseau. Les petits écarts entre deux réseaux (quelques €/m²) peuvent dépendre de la graine.

### 5.3 Référence plus juste pour l'évaluation B : régression hédonique apprise sur N-1

En évaluation B, la naïve donne le même prix à toutes les ventes d'un sous-bassin (le prix médian de N-1), alors que les modèles connaissent la surface, le terrain et le nombre de pièces. Une partie du gain sur la naïve vient donc de l'information sur le bien, pas de la prévision du marché. La référence ci-dessous apprend, sur les seules ventes de N-1, le prix au m² en fonction du bien et du sous-bassin, puis l'applique aux ventes de N : c'est la naïve corrigée du bien.

- **Modèle :** régression log-linéaire régularisée (Elastic Net), mêmes variables que l'Elastic Net du « où » : le sous-bassin et le bloc du bien. Ni prix N-1, ni territoire, ni indicateur du « quand ».
- **Apprentissage :** les ventes de N-1 seules. Le réglage (alpha, l1_ratio) est choisi en apprenant sur N-2 et en jugeant sur N-1 : aucune vente de N n'intervient.
- **Années jugées :** 2023 et 2024. 2025 n'est pas touchée.

In [35]:
VARIABLES_HEDONIQUE = ["sous_bassin_de_vie"] + BLOC_BIEN
HEDONIQUE = "Hédonique N-1 – Elastic Net – sous-bassin + bien"


def prevoir_hedonique(app, ev, variables):
    """Elastic Net appris sur les ventes de N-1 seules. Réglage : apprendre sur N-2, juger sur N-1."""
    colonnes = list(variables) + ["annee"]
    n1 = app["annee"].max()                                   # dernière année d'apprentissage de la fenêtre = N-1
    avant, derniere = app[app["annee"] == n1 - 1], app[app["annee"] == n1]
    essais = []
    for g in ParameterGrid(GRILLE_EN):
        m = ElasticNetVentes(**g).fit(avant[colonnes], avant["prix_m2"], sample_weight=avant["w_h"])
        mae = np.average((derniere["prix_m2"] - m.predict(derniere[colonnes])).abs(), weights=derniere["w_h"])
        essais.append({**g, "MAE": mae})
    meilleur = min(essais, key=lambda e: e["MAE"])
    modele = ElasticNetVentes(**{k: meilleur[k] for k in GRILLE_EN}).fit(
        derniere[colonnes], derniere["prix_m2"], sample_weight=derniere["w_h"])
    return modele.predict(ev[colonnes]), modele


# Relancer la cellule remplace les résultats au lieu de les ajouter une seconde fois
resultats[:] = [r for r in resultats if not r["modèle"].startswith(HEDONIQUE + " (")]
tests[:] = [t for t in tests if not t["modèle"].startswith(HEDONIQUE + " (")]

for annee, (app, ev) in FENETRES.items():
    assert annee in commun.ANNEES_JUGEES and ev["annee"].max() < commun.ANNEE_TEST, "2025 ne doit pas être touchée"
    panier = panier_fixe(ventes_brutes, table, annee)
    prevision, m = prevoir_hedonique(app, ev, VARIABLES_HEDONIQUE)
    prevision_panier = m.predict(panier)
    ajustes[(HEDONIQUE, annee)] = m
    previsions[(HEDONIQUE, annee)] = (prevision, prevision_panier)
    evaluer_ventes(HEDONIQUE, ev, prevision, resultats)
    tester_ventes(HEDONIQUE, ev, prevision, tests)
    evaluer_panier(HEDONIQUE, panier, prevision_panier, table, resultats, tests)
    print(f"Fenêtre {annee} : appris sur {annee - 1} | alpha {m.alpha:.0e}, l1_ratio {m.l1_ratio}")

res_h = pd.DataFrame(resultats)
res_h = res_h[res_h["modèle"].str.startswith(HEDONIQUE + " (")][
    ["modèle", "type de bien", "année jugée", "MAE (€/m²)", "MAE naïve (€/m²)",
     "gain sur la naïve (%)", "bassins où le modèle gagne (%)"]]
res_h["évaluation"] = res_h.pop("modèle").str.extract(r"\((B|C)", expand=False)
print(res_h.round(1).to_string(index=False))

Fenêtre 2023 : appris sur 2022 | alpha 3e-03, l1_ratio 0.8
Fenêtre 2024 : appris sur 2023 | alpha 3e-03, l1_ratio 0.8
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       761.4             732.6                   -3.9                            47.3          B
      maison         2023       800.4             774.2                   -3.4                            48.1          B
 appartement         2023       327.2             166.2                  -96.9                            25.6          C
      maison         2023       288.6             111.7                 -158.3                            22.6          C
 appartement         2024       790.7             744.9                   -6.1                            49.6          B
      maison         2024       773.4             740.6                   -4.4                            48.5          B
 appartement         2024   

### 5.4 La référence hédonique comme comparaison

Même méthode qu'en 5.2 (mêmes mesures, mêmes tests, même règle de décision), avec la prévision de la référence hédonique à la place de la naïve. Question : le gain des réseaux tient-il à ce qu'on les compare à la naïve, ou résiste-t-il à une référence qui connaît le bien et que l'on apprend sur N-1 seulement ?

In [39]:
COMPARAISONS_HEDO = [  # (libellé, modèle, modèle de comparaison)
    ("H7v1 contre la référence hédonique N-1", H7V1, HEDONIQUE),
    ("Elastic Net du « quand » contre la référence hédonique N-1", REFERENCE_QUAND, HEDONIQUE),
    ("H7v3 contre la référence hédonique N-1", H7V3, HEDONIQUE),
    ("H7v4 contre la référence hédonique N-1", H7V4, HEDONIQUE),
]

resultats_hedo, tests_hedo = [], []
for libelle, nom, nom_ref in COMPARAISONS_HEDO:
    for annee, (app, ev) in FENETRES.items():
        assert annee in commun.ANNEES_JUGEES and ev["annee"].max() <= commun.ANNEES_JUGEES[-1]   # jamais 2025     # jamais 2025
        p, p_panier = previsions[(nom, annee)]
        r, r_panier = previsions[(nom_ref, annee)]
        ev_ref = ev.copy()
        ev_ref["prix_n1"] = r                                    # B : la référence remplace la naïve
        evaluer_ventes(libelle, ev_ref, p, resultats_hedo)
        tester_ventes(libelle, ev_ref, p, tests_hedo)
        panier = panier_fixe(ventes_brutes, table, annee)        # C : médiane des prévisions du panier fixe
        med, med_ref = medianes_panier(panier, p_panier), medianes_panier(panier, r_panier)
        evaluation = table.loc[table.index.intersection(med.index)].copy()
        evaluation["prix_n1"] = med_ref.loc[evaluation.index]
        evaluer(f"{libelle} (C, panier fixe)", evaluation, med.loc[evaluation.index], resultats_hedo)
        tester(f"{libelle} (C, panier fixe)", evaluation, med.loc[evaluation.index], tests_hedo)

res_hedo = pd.DataFrame(resultats_hedo).rename(columns=lambda c: c.replace("naïve", "référence"))
tst_hedo = pd.DataFrame(tests_hedo).rename(columns=lambda c: c.replace("naïve", "référence"))
display(res_hedo[["modèle", "type de bien", "année jugée", "MAE (€/m²)", "MAE référence (€/m²)",
                  "gain sur la référence (%)", "bassins où le modèle gagne (%)"]].round(2))
display(regle_decision(res_hedo, tst_hedo, comparaison="référence"))

,modèle,type de bien,année jugée,MAE (€/m²),MAE référence (€/m²),gain sur la référence (%),bassins où le modèle gagne (%)
0,"H7v1 contre la référence hédonique N-1 (B, ven...",appartement,2023,802.79,761.42,-5.43,44.69
1,"H7v1 contre la référence hédonique N-1 (B, ven...",maison,2023,791.57,800.43,1.11,57.92
2,"H7v1 contre la référence hédonique N-1 (C, pan...",appartement,2023,454.59,327.22,-38.93,42.01
3,"H7v1 contre la référence hédonique N-1 (C, pan...",maison,2023,342.51,288.57,-18.69,45.73
4,"H7v1 contre la référence hédonique N-1 (B, ven...",appartement,2024,803.89,790.67,-1.67,44.50
5,"H7v1 contre la référence hédonique N-1 (B, ven...",maison,2024,728.99,773.38,5.74,63.34
6,"H7v1 contre la référence hédonique N-1 (C, pan...",appartement,2024,417.41,339.88,-22.81,43.73
7,"H7v1 contre la référence hédonique N-1 (C, pan...",maison,2024,256.05,278.18,7.96,53.83
8,Elastic Net du « quand » contre la référence h...,appartement,2023,718.04,761.42,5.70,63.44
9,Elastic Net du « quand » contre la référence h...,maison,2023,756.01,800.43,5.55,62.92


années réussies  années jugées  retenu
modèle                                             type de bien                                        
H7v1 contre la référence hédonique N-1 (B, ventes) appartement                 0              2   False
                                                   maison                      1              2   False
H7v1 contre la référence hédonique N-1 (C, pani... appartement                 0              2   False
                                                   maison                      1              2   False
Elastic Net du « quand » contre la référence hé... appartement                 2              2    True
                                                   maison                      2              2    True
Elastic Net du « quand » contre la référence hé... appartement                 2              2    True
                                                   maison                      2              2    True
H7v3 contre la référence hédonique N-1 (B, ventes) appartement                 2              2    True
                                                   maison                      2              2    True
H7v3 contre la référence hédonique N-1 (C, pani... appartement                 2              2    True
                                                   maison                      1              2   False
H7v4 contre la référence hédonique N-1 (B, ventes) appartement                 2              2    True
                                                   maison                      2              2    True
H7v4 contre la référence hédonique N-1 (C, pani... appartement                 2              2    True
                                                   maison                      2              2    True

### 5.5 Évaluation C avec un panier apparié

Le panier fixe prédit les biens vendus en N-1, alors que le prix réel de N est la médiane des biens vendus en N : les deux ensembles ne sont pas composés des mêmes biens. Le panier apparié prédit les ventes réelles de N : la prévision et le prix réel portent sur les mêmes biens. Si la composition expliquait l'écart de l'évaluation C, il devrait se réduire. Comparaison à la naïve, sur 2023 et 2024 seulement.

In [40]:
MODELES_APPARIES = [H7V4, H7V3, REFERENCE_QUAND, HEDONIQUE]

resultats_c, tests_c = [], []
for nom in MODELES_APPARIES:
    for annee, (app, ev) in FENETRES.items():
        assert annee in commun.ANNEES_JUGEES and ev["annee"].max() <= commun.ANNEES_JUGEES[-1]   # jamais 2025
        p, p_panier = previsions[(nom, annee)]
        panier = panier_fixe(ventes_brutes, table, annee)
        for etiquette, base, prev in (("panier fixe", panier, p_panier), ("panier apparié", ev, p)):
            med = medianes_panier(base, prev)
            evaluation = table.loc[table.index.intersection(med.index)].copy()
            nom_c = f"{nom} (C, {etiquette})"
            evaluer(nom_c, evaluation, med.loc[evaluation.index], resultats_c)
            tester(nom_c, evaluation, med.loc[evaluation.index], tests_c)

res_c = pd.DataFrame(resultats_c)
res_c["panier"] = res_c["modèle"].str.extract(r"\(C, (.*)\)")[0]
res_c["modèle"] = res_c["modèle"].str.replace(r" \(C, .*\)", "", regex=True)
display(res_c.pivot_table(index=["modèle", "type de bien", "année jugée"], columns="panier",
                          values=["MAE (€/m²)", "gain sur la naïve (%)"], aggfunc="first").round(1))
display(regle_decision(pd.DataFrame(resultats_c), pd.DataFrame(tests_c)))

MAE (€/m²)             gain sur la naïve (%)            
panier                                                                      panier apparié panier fixe        panier apparié panier fixe
modèle                                             type de bien année jugée                                                             
H7 – Elastic Net – sous-bassin + bien + territo... appartement  2023                 219.4       217.8                 -32.1       -31.1
                                                                2024                 172.2       179.3                 -17.1       -21.9
                                                   maison       2023                 154.9       155.2                 -38.7       -38.9
                                                                2024                 121.1       120.7                   6.3         6.5
H7v3 – Keras – sous-bassin + bien + territoire ... appartement  2023                 211.4       211.9                 -27.3       -27.5
                                                                2024                 228.3       235.4                 -55.2       -60.0
                                                   maison       2023                 200.1       195.1                 -79.1       -74.6
                                                                2024                 255.1       253.9                 -97.4       -96.5
H7v4 – Keras – H7v3 + volume, élan, capacité d'... appartement  2023                 196.1       199.5                 -18.1       -20.0
                                                                2024                 186.8       193.9                 -27.1       -31.8
                                                   maison       2023                 190.3       184.6                 -70.4       -65.3
                                                                2024                 152.8       157.3                 -18.2       -21.8
Hédonique N-1 – Elastic Net – sous-bassin + bien   appartement  2023                 322.3       327.2                 -94.1       -96.9
                                                                2024                 332.1       339.9                -125.8      -131.1
                                                   maison       2023                 286.2       288.6                -156.2      -158.3
                                                                2024                 274.0       278.2                -112.0      -115.3

années réussies  années jugées  retenu
modèle                                             type de bien                                        
H7v4 – Keras – H7v3 + volume, élan, capacité d'... appartement                 0              2   False
                                                   maison                      0              2   False
H7v4 – Keras – H7v3 + volume, élan, capacité d'... appartement                 0              2   False
                                                   maison                      0              2   False
H7v3 – Keras – sous-bassin + bien + territoire ... appartement                 0              2   False
                                                   maison                      0              2   False
H7v3 – Keras – sous-bassin + bien + territoire ... appartement                 0              2   False
                                                   maison                      0              2   False
H7 – Elastic Net – sous-bassin + bien + territo... appartement                 0              2   False
                                                   maison                      1              2   False
H7 – Elastic Net – sous-bassin + bien + territo... appartement                 0              2   False
                                                   maison                      1              2   False
Hédonique N-1 – Elastic Net – sous-bassin + bie... appartement                 0              2   False
                                                   maison                      0              2   False
Hédonique N-1 – Elastic Net – sous-bassin + bie... appartement                 0              2   False
                                                   maison                      0              2   False

## 6. Interprétation

### 6.1 Erreurs de H7v3 selon le niveau des prix du sous-bassin
Évaluation B, par année jugée et type de bien : les sous-bassins sont répartis en quatre groupes de même taille selon leur prix N-1 (quartiles), du moins cher au plus cher. Par groupe : erreur moyenne du modèle et de la naïve, écart moyen (négatif : le modèle fait mieux) et part des sous-bassins où le modèle gagne. Puis, pour les appartements en 2024 (seul cas où H7v3 n'est pas retenu contre la naïve en évaluation B), les dix sous-bassins où le modèle perd le plus.

In [36]:
from commun import CLES, _par_sous_bassin

QUARTILES = ["Q1 (moins chers)", "Q2", "Q3", "Q4 (plus chers)"]
for annee, (app, ev) in FENETRES.items():
    t = _par_sous_bassin(ev, previsions[(H7V3, annee)][0])        # erreurs pondérées par w_h, par sous-bassin
    t["écart (€/m²)"] = t["modele"] - t["naive"]
    t["prix N-1 (€/m²)"] = table["prix_n1"].reindex(pd.MultiIndex.from_frame(t[table.index.names])).to_numpy()
    t["ventes"] = ev.groupby(CLES).size().reindex(pd.MultiIndex.from_frame(t[CLES])).to_numpy()
    for type_bien, e in t.groupby("type_bien"):
        quartile = pd.qcut(e["prix N-1 (€/m²)"], 4, labels=QUARTILES)
        resume = e.groupby(quartile, observed=True).agg(
            **{"sous-bassins": ("écart (€/m²)", "size"),
               "prix N-1 médian (€/m²)": ("prix N-1 (€/m²)", "median"),
               "MAE modèle (€/m²)": ("modele", "mean"), "MAE naïve (€/m²)": ("naive", "mean"),
               "écart moyen (€/m²)": ("écart (€/m²)", "mean"),
               "sous-bassins où le modèle gagne (%)": ("écart (€/m²)", lambda x: (x < 0).mean() * 100)})
        print(f"\n{annee} – {type_bien}")
        print(resume.round(1).to_string())

# Les 10 sous-bassins où H7v3 perd le plus contre la naïve : appartements, 2024 (t contient la dernière fenêtre)
print("\nAppartements 2024 : les 10 sous-bassins où H7v3 perd le plus contre la naïve")
pire = t[t["type_bien"] == "appartement"].nlargest(10, "écart (€/m²)")
print(pire[["sous_bassin_de_vie", "BV2022", "ventes", "prix N-1 (€/m²)", "modele", "naive", "écart (€/m²)"]]
      .rename(columns={"modele": "MAE modèle (€/m²)", "naive": "MAE naïve (€/m²)"}).round(0).to_string(index=False))


2023 – appartement
                  sous-bassins  prix N-1 médian (€/m²)  MAE modèle (€/m²)  MAE naïve (€/m²)  écart moyen (€/m²)  sous-bassins où le modèle gagne (%)
prix N-1 (€/m²)                                                                                                                                     
Q1 (moins chers)           212                  1292.0              364.2             385.5               -21.3                                 61.8
Q2                         212                  1953.5              526.3             548.2               -21.9                                 62.3
Q3                         212                  2731.0              635.5             687.6               -52.1                                 67.0
Q4 (plus chers)            212                  4199.5             1268.5            1309.2               -40.7                                 61.3

2023 – maison
                  sous-bassins  prix N-1 médian (€/m²)  MAE modèle (€/m

**Résultats :**
- H7v3 bat la naïve dans tous les quartiles de prix, pour les deux types de bien et les deux années. Pour les maisons, l'écart va de −31 à −66 €/m² et le modèle gagne dans 58 à 76 % des sous-bassins ; pour les appartements, de −15 à −54 €/m², dans 61 à 70 % des sous-bassins.
- Le gain ne dépend pas nettement du niveau des prix : il est un peu plus fort dans les quartiles intermédiaires et chers (Q3 et Q4) que dans le quartile le moins cher, mais la part de sous-bassins gagnés reste du même ordre partout. Pour les maisons, elle baisse dans le quartile le plus cher (63 % en 2023, 58 % en 2024, contre 73 à 76 % dans le moins cher).
- Les dix sous-bassins où H7v3 perd le plus contre la naïve (appartements 2024, écart de +278 à +602 €/m²) sont de petits sous-bassins, avec 1 à 16 ventes jugées, dont six avec une ou deux ventes. Leur prix N-1 va de 1 600 à 5 500 €/m² : aucun n'est un marché extrême.
- Hypothèse : sur une ou deux ventes, l'erreur dépend surtout du bien vendu, et la médiane N-1 du sous-bassin peut tomber juste par hasard. Ces pertes relèvent du bruit des petits effectifs plutôt que d'une faiblesse du modèle sur un type de marché.

### 6.2 Importance des blocs de variables (H7v3)
Importance par permutation, sur l'année jugée, par type de bien : on mélange au hasard les lignes d'un bloc de variables entre les ventes (toutes les colonnes du bloc ensemble, par exemple les cinq indicatrices du nombre de pièces), 5 fois, et on mesure de combien l'erreur moyenne par vente augmente (pondérée par `w_h`). Une hausse nette : le modèle s'appuie sur ce bloc ; proche de 0 : il ne s'en sert pas. Une hausse se lit avec son écart-type : un bloc compte si elle le dépasse nettement.

Limite : deux blocs qui portent la même information (le sous-bassin et le prix N-1 disent tous deux le niveau des prix du lieu) se partagent l'importance ; mélanger l'un laisse l'autre en place, et l'importance de chacun est sous-estimée.

In [37]:
BLOCS = {"sous-bassin (embedding)": ["sous_bassin_de_vie"], "surface": ["log_sbati"], "terrain": ["log1p_sterr"],
         "pièces": COLONNES_PIECES, "dépendances": COLONNES_DEP, "territoire": BLOC_TERRITOIRE,
         "prix N-1": ["log_prix_n1"]}
N_REPETITIONS = 5
rng = np.random.default_rng(commun.GRAINE)

importances = []
for annee, (app, ev) in FENETRES.items():
    m = ajustes[(H7V3, annee)]
    for type_bien, e in ev.groupby("type_bien"):
        def mae(X):
            return np.average((e["prix_m2"] - m.predict(X)).abs(), weights=e["w_h"])
        base = mae(e)
        for bloc, colonnes in BLOCS.items():
            hausses = []
            for _ in range(N_REPETITIONS):
                X = e.copy()
                X[colonnes] = e[colonnes].to_numpy()[rng.permutation(len(e))]   # mêmes lignes mélangées pour tout le bloc
                hausses.append(mae(X) - base)
            importances.append({"année jugée": annee, "type de bien": type_bien, "bloc": bloc,
                                "hausse de la MAE (€/m²)": np.mean(hausses), "écart-type (€/m²)": np.std(hausses)})

importances = pd.DataFrame(importances)
importances.pivot_table(index="bloc", columns=["type de bien", "année jugée"],
                        values=["hausse de la MAE (€/m²)", "écart-type (€/m²)"]).round(1)

hausse de la MAE (€/m²)                       écart-type (€/m²)                 
type de bien                        appartement         maison              appartement      maison     
année jugée                                2023    2024   2023   2024              2023 2024   2023 2024
bloc                                                                                                    
dépendances                                16.7    15.6   10.1    9.6               0.9  1.1    0.4  0.5
pièces                                     99.3    95.2   12.8   13.8               2.0  0.5    0.3  0.7
prix N-1                                 1556.2  1281.1  847.9  686.0               7.8  4.0    2.0  5.0
sous-bassin (embedding)                    17.8    23.6   12.7   16.3               0.9  0.9    0.9  0.9
surface                                   256.4   226.4  107.5  108.2               2.1  3.4    0.9  1.7
terrain                                     0.0     0.0   91.5   94.7               0.0  0.0    0.8  1.7
territoire                                 63.2    71.1   18.8   30.4               1.4  1.2    0.8  1.0

**Comment lire :** prenons la surface, appartements, 2023 : la hausse vaut 256,4 €/m² avec un écart-type de 2,1. Quand on mélange les surfaces entre les ventes, l'erreur moyenne par vente augmente de 256 €/m² ; la hausse dépasse très nettement l'écart-type : le modèle s'appuie bien sur la surface.

**Résultats :**

- Le prix N-1 porte l'essentiel de la prévision : le mélanger fait monter l'erreur de 686 à 1 556 €/m² selon l'année et le type de bien. Son poids est plus fort pour les appartements (1 281 et 1 556 €/m²) que pour les maisons (686 et 848 €/m²).
- Le bien vient ensuite. La surface est le bloc le plus utile après le prix N-1 (226 et 256 €/m² pour les appartements, 108 pour les maisons) ; pour les maisons, le terrain compte presque autant que la surface (92 et 95 €/m²), ce qu'annonçait le constat (section 2.2). Le nombre de pièces compte pour les appartements (95 et 99 €/m²), peu pour les maisons (13 et 14 €/m²) ; les dépendances apportent peu (10 à 17 €/m²).
- Le territoire (19 à 71 €/m²) compte plus que le sous-bassin (13 à 24 €/m²), surtout pour les appartements : les deux portent la même information sur le lieu, et le réseau s'appuie surtout sur le territoire. C'est cohérent avec H7v2b, où le territoire décrivait une bonne part du lieu sans remplacer l'identité du sous-bassin.
- Les écarts-types restent faibles (8 €/m² au plus) : le classement des blocs est stable d'un mélange à l'autre.
- Le prix N-1 dit lui aussi le niveau des prix du lieu : l'importance du sous-bassin et du territoire est donc sous-estimée (limite de la méthode).

## 7. Conclusion

**Question :** un modèle par vente sépare-t-il l'effet du bien et celui du marché, et fait-il mieux que les modèles agrégés à information égale ? Avec les indicateurs du niveau 2 (volume et élan, capacité d'achat, trimestres précédents), anticipe-t-il mieux le retournement ?

**1. Le lieu et le bien ne suffisent pas.** Sans information sur le moment (Elastic Net du « où », H7v1, H7v1b, H7v2a, H7v2b), aucun modèle n'est retenu contre la naïve, ni pour chaque vente (évaluation B) ni pour la médiane d'un sous-bassin (évaluation C) ; seul H7v2a réussit une année (maisons 2024, gain de 3,1 %). Ces modèles sous-estiment les prix de 2 à 11 % : ils apprennent le niveau moyen de la période 2017 à l'année d'avant, alors que les prix ont fortement monté sur cette période. À informations égales, le réseau fait nettement mieux que l'Elastic Net (H7v1 retenu contre lui en évaluation B pour les deux types de bien, et en évaluation C pour les maisons). Les coordonnées de la parcelle n'apportent presque rien au sous-bassin (gain de 1,3 % au plus) ; les variables de territoire du niveau 1 (parc, vacance, densité) ajoutent peu au sous-bassin et ne le remplacent pas (H7v2b fait moins bien que H7v1 partout).

**2. Le prix N-1 change tout.** Ajouté à H7v2a, il réduit l'erreur de 11 à 13 % par vente et de 41 à 54 % sur la médiane d'un sous-bassin pour les appartements, les deux années ; pour les maisons, nettement en 2023 (8 % et 44 %), à peine en 2024. Avec lui, les réseaux battent la naïve pour prévoir le prix d'un bien vendu, pour les deux types de bien et les deux années : H7v3, H7vH5, H7vH6, H7vT et H7v4 sont retenus en évaluation B (gain de 4,6 à 7,3 %, IC entièrement sous 0, rang-bisériale de −0,34 à −0,54). H7v3 bat la naïve dans tous les quartiles de prix ; ses plus fortes pertes sont dans de petits sous-bassins (1 à 16 ventes jugées), où l'erreur relève surtout du hasard. Le prix N-1 inverse aussi le biais : les réseaux surestiment désormais les prix (H7v3 : de +9 à +14 %), en prolongeant la hausse apprise sur 2017-2022.

**3. Le modèle sépare bien l'effet du bien.** Après le prix N-1, la surface est le bloc le plus utile, puis le terrain pour les maisons et le nombre de pièces pour les appartements ; les dépendances apportent peu. C'est ce qui permet au modèle de battre la naïve vente par vente : la naïve donne le même prix à toutes les ventes d'un sous-bassin.

**4. Réseau ou Elastic Net : chacun son usage.** Avec le prix N-1, le réseau fait mieux que l'Elastic Net pour chaque vente (IC sous 0 dans les quatre cas), mais avec un gain de 2,2 à 4,1 % qui ne dépasse le seuil qu'une année par type de bien : il n'est pas retenu contre lui. Pour la médiane d'un sous-bassin, l'Elastic Net fait nettement mieux dans trois cas sur quatre, car il est moins biaisé (+3,5 à +5,9 % contre +9 à +14 %). C'est le seul modèle de H7 qui bat la naïve une année en évaluation C (maisons 2024 : gain de 6,5 %), sans être retenu (une année sur deux).

**5. Les indicateurs du « quand » corrigent le niveau, pas l'erreur de chaque vente.** Contre H7v3, aucun n'est retenu en évaluation B (gain de 0 à 2,6 %). En évaluation C, ils réduisent l'erreur sur la médiane d'un sous-bassin : la capacité d'achat (H7vH6) et tous les indicateurs réunis (H7v4) sont retenus pour les deux types de bien. Ils agissent par le biais de 2024 : de +9,6 % (appartements) et +13,7 % (maisons) pour H7v3, il passe à +2,3 et +1,8 % avec la capacité d'achat, à +1,3 et +3,8 % avec H7v4. Le taux d'effort, qui suit la remontée des taux des crédits, porte l'essentiel de cette correction. En 2023, le biais reste élevé (de +5,6 à +9,2 %) : le signal de fin 2022 était faible (prix encore en hausse de 6 à 9 %, volume en baisse modérée), alors que celui de fin 2023 était net (volume en baisse de 20 à 27 %, prix du 4e trimestre déjà en baisse).

**6. Il ne prévoit pas mieux le marché.** Pour la médiane d'un sous-bassin (évaluation C, la comparaison avec les modèles agrégés), aucun réseau ne bat la naïve, même corrigé de son biais (H7v4 : de −20 à −65 %). La comparaison chiffrée avec les modèles agrégés (H4 à H6) est faite dans le rapport.

**Candidat pour le test final 2025 : H7v4.** Cinq réseaux sont retenus contre la naïve en évaluation B. H7v4 est choisi parce qu'il est, avec H7vH6, le meilleur en évaluation B (gain de 5,4 à 7,3 %), qu'il est retenu contre H7v3 en évaluation C pour les deux types de bien, et qu'il a le plus faible biais de 2024 pour les appartements (+1,3 %). Il réunit tous les indicateurs du « quand », sans grand écart avec H7vH6.

**Limites :**
- Le bootstrap mesure le hasard dû aux territoires, pas celui de l'entraînement du réseau : les petits écarts entre deux réseaux (quelques €/m²) peuvent dépendre de la graine.
- De nombreux modèles sont comparés sans correction des comparaisons multiples.
- Les modèles apprennent sur l'échantillon de ventes (environ 10 % des ventes par département et par année, pondéré par `w_h`), pas sur toutes les ventes.
- Les variables de territoire sont des photos fixes, prises à un seul millésime pour toute la période.
- Un seul retournement dans les données (2023-2024) : aucun modèle ne peut apprendre à l'anticiper à partir d'un seul exemple.
- Les trimestres sont calculés au département, avec la valeur nationale pour 3,1 à 4,4 % des ventes ; l'élan inconnu est mis à 0 (0,3 % des ventes d'appartements au plus).